## XGBoost LOO analysis: title-free figures and PDF export

This notebook uses only the XGBoost results in `xgb_res`. Run all cells in order with the original XGBoost data files beside the notebook. Plot titles are removed; the larger fonts, axis labels, legends, and existing analysis plots are retained.

The final plotting section adds one horizontal **1 × 3** protein-wise figure: **PCC, MSE, and cosine similarity**. It reuses the same per-protein metric table as the earlier detailed performance plot, so its evaluation set and values match that plot. Summary statistics and best/worst protein lists are printed as separate tables and saved as CSV files.

The last cell saves every generated plot as a PDF in **`figures_pdf` beside this notebook**, including figures already closed. The final compact figure has the fixed filename **`xgboost_loo_protein_metrics_1x3.pdf`**; table CSVs are in `figures_pdf/tables`. Restart the kernel before Run All to clear figures from previous runs. If your kernel starts elsewhere, set `NOTEBOOK_DIRECTORY_OVERRIDE` in the setup cell.


In [ ]:
# >>> pRNA figure routing (added by the pRNA update; keep as the first code cell)
# Every figure is shown inline. Every file written with savefig (png, pdf, svg) is redirected to
# figures_pdf/<notebook name>/ next to this notebook; sub-folders of the original path are preserved.
import os as _fig_os
from pathlib import Path as _FigPath
import matplotlib as _fig_mpl
import matplotlib.pyplot as _fig_plt
from matplotlib.figure import Figure as _FigFigure

NOTEBOOK_NAME = "02_res_analysis_loo_xgb"
FIG_DIR = (_FigPath.cwd() / "figures_pdf" / NOTEBOOK_NAME).resolve()
FIG_DIR.mkdir(parents=True, exist_ok=True)


def _route_figure_path(fname):
    p = _FigPath(_fig_os.fspath(fname))
    full = p if p.is_absolute() else (_FigPath.cwd() / p)
    try:
        full.resolve().relative_to(FIG_DIR)
        return fname                                  # already inside this notebook's figure folder
    except ValueError:
        pass
    try:
        rel = _FigPath(_fig_os.path.relpath(full.resolve(), _FigPath.cwd()))
    except ValueError:                                # e.g. another drive on Windows
        rel = _FigPath(p.name)
    parts = [x for x in rel.parts if x not in ("..", ".", "figures_pdf", NOTEBOOK_NAME)]
    target = FIG_DIR.joinpath(*parts) if parts else FIG_DIR / p.name
    target.parent.mkdir(parents=True, exist_ok=True)
    return target


if not getattr(_FigFigure.savefig, "_prna_routed", False):
    _fig_orig_savefig = _FigFigure.savefig

    def _fig_routed_savefig(self, fname, *args, **kwargs):
        if isinstance(fname, (str, bytes, _FigPath)) or hasattr(fname, "__fspath__"):
            fname = _route_figure_path(fname)
        return _fig_orig_savefig(self, fname, *args, **kwargs)

    _fig_routed_savefig._prna_routed = True
    _FigFigure.savefig = _fig_routed_savefig

    _fig_orig_close = _fig_plt.close

    def _fig_close_and_show(fig=None):
        """Display figures that are closed before Jupyter could render them, then close."""
        import inspect as _fig_inspect
        _caller = _fig_inspect.currentframe().f_back
        _caller_mod = (_caller.f_globals.get("__name__", "") if _caller is not None else "")
        if _caller_mod.startswith(("matplotlib", "ipykernel")):   # backend housekeeping, not user code
            return _fig_orig_close(fig)
        if "inline" in str(_fig_mpl.get_backend()).lower():
            from IPython.display import display as _fig_display
            open_nums = set(_fig_plt.get_fignums())
            if fig is None:
                figs = [_fig_plt.gcf()] if open_nums else []
            elif isinstance(fig, str) and fig == "all":
                figs = [_fig_plt.figure(n) for n in sorted(open_nums)]
            elif isinstance(fig, _FigFigure):
                figs = [fig] if getattr(fig, "number", None) in open_nums else []
            elif isinstance(fig, int):
                figs = [_fig_plt.figure(fig)] if fig in open_nums else []
            else:
                figs = []
            for _f in figs:
                if not getattr(_f, "_prna_displayed", False):
                    _f._prna_displayed = True
                    _fig_display(_f)
        return _fig_orig_close(fig)

    _fig_plt.close = _fig_close_and_show

    _fig_orig_show = _fig_plt.show

    def _fig_show_and_mark(*args, **kwargs):
        for _n in _fig_plt.get_fignums():                    # shown now -> never re-displayed on close
            _fig_plt.figure(_n)._prna_displayed = True
        return _fig_orig_show(*args, **kwargs)

    _fig_plt.show = _fig_show_and_mark

print("Figures are saved to", FIG_DIR)

In [ ]:
# Larger plot text and deferred PDF export.
# Keep this cell before the analysis/plotting cells.
from pathlib import Path
from collections import OrderedDict
from io import BytesIO
import os
import re
import textwrap
import warnings
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib._pylab_helpers import Gcf
from matplotlib.ticker import MaxNLocator

NOTEBOOK_SLUG = 'loo_xgb'
REMOVE_PLOT_TITLES = True
NOTEBOOK_DIRECTORY_OVERRIDE = None  # Set a Path here only if the kernel runs elsewhere.


def _notebook_directory():
    if NOTEBOOK_DIRECTORY_OVERRIDE is not None:
        return Path(NOTEBOOK_DIRECTORY_OVERRIDE).expanduser().resolve()
    # VS Code and Jupyter may provide the notebook's actual path.
    candidates = [globals().get('__vsc_ipynb_file__'), os.environ.get('JPY_SESSION_NAME')]
    for candidate in candidates:
        if candidate:
            path = Path(candidate).expanduser()
            if path.suffix == '.ipynb' and path.is_file():
                return path.resolve().parent
    try:
        import ipynbname
        return ipynbname.path().resolve().parent
    except (ImportError, FileNotFoundError, RuntimeError, OSError):
        # Standard Jupyter kernels start in the notebook's own directory.
        return Path.cwd().resolve()


NOTEBOOK_DIRECTORY = _notebook_directory()
FIGURES_PDF_DIR = FIG_DIR
_pdf_figure_cache = OrderedDict()
_pdf_next_number = 1

mpl.rcParams.update({
    'font.size': 13,
    'axes.titlesize': 17,
    'axes.labelsize': 14,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'legend.fontsize': 12,
    'legend.title_fontsize': 12,
    'figure.titlesize': 22,
    'figure.figsize': (8, 6.5),
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.25,
})


def set_readable_theme(*args, **kwargs):
    """Keep Seaborn's own implicit label/tick fonts larger as well."""
    import seaborn as sns
    kwargs.setdefault('font_scale', 1.20)
    return sns.set_theme(*args, **kwargs)


def readable_figsize(original_size):
    """Increase physical space together with fonts, including dynamic grids."""
    width, height = original_size
    if NOTEBOOK_SLUG == 'hpa_barplots':
        return float(width) * 1.18, float(height) * 1.35
    return float(width) * 1.10, float(height) * 1.25


def _wrap_to_width(artist, renderer, max_width):
    """Wrap whole words using measured text extents, without shortening labels."""
    original = getattr(artist, '_readable_original_text', artist.get_text())
    artist._readable_original_text = original
    if not original or '$' in original:
        return
    artist.set_text(original)
    if artist.get_window_extent(renderer).width <= max_width:
        return
    for width in range(max(12, max(map(len, original.splitlines())) - 1), 11, -1):
        candidate = '\n'.join(
            textwrap.fill(line, width=width, break_long_words=False, break_on_hyphens=False)
            for line in original.split('\n')
        )
        artist.set_text(candidate)
        if artist.get_window_extent(renderer).width <= max_width:
            break



def _remember_figure_name(fig, text):
    """Keep descriptive export names without displaying a plot title."""
    fig._readable_export_title = str(text)


def _remember_axes_name(ax, text):
    if not hasattr(ax.figure, '_readable_export_axes_title'):
        ax.figure._readable_export_axes_title = str(text)


def prepare_readable_figure(fig):
    """Apply minimum font sizes and measure margins before display or export."""
    if not fig.axes or getattr(fig, '_readable_preparing', False):
        return fig
    fig._readable_preparing = True
    try:
        if REMOVE_PLOT_TITLES:
            for axis in fig.axes:
                for location in ('left', 'center', 'right'):
                    title_artist = axis.set_title('', loc=location)
                    title_artist._readable_original_text = ''
            if fig._suptitle is not None:
                fig._suptitle.remove()
                fig._suptitle = None
        if getattr(fig, '_compact_protein_metrics', False):
            # Short, wide layout for stacking five panels on an A4 page.
            fig.tight_layout(pad=0.25, w_pad=0.8, h_pad=0.25)
            fig.canvas.draw()
            return fig
        for ax in fig.axes:
            if not ax.get_visible():
                continue
            ax.title.set_fontsize(max(16, ax.title.get_fontsize()))
            ax.xaxis.label.set_fontsize(max(14, ax.xaxis.label.get_fontsize()))
            ax.yaxis.label.set_fontsize(max(14, ax.yaxis.label.get_fontsize()))
            ax.xaxis.labelpad = max(8, ax.xaxis.labelpad)
            ax.yaxis.labelpad = max(9, ax.yaxis.labelpad)
            ax.tick_params(axis='both', which='major', pad=5)
            for label in ax.get_xticklabels() + ax.get_yticklabels():
                label.set_fontsize(max(11.5, label.get_fontsize()))
            for offset in (ax.xaxis.get_offset_text(), ax.yaxis.get_offset_text()):
                offset.set_fontsize(max(11.5, offset.get_fontsize()))
            for artist in ax.texts:
                artist.set_fontsize(max(11.5, artist.get_fontsize()))
                if re.fullmatch(r'\(?[A-Za-z]\)', artist.get_text().strip()):
                    # Panel letters sit outside the data and statistics boxes.
                    artist.set_position((-0.06, 1.03))
                    artist.set_ha('right')
                    artist.set_va('bottom')
                    artist.set_clip_on(False)
                if not ax.axison and artist.get_text().count('\n') >= 8:
                    # Keep the full statistics box within its own panel.
                    artist.set_position((0.03, 0.96))
                    artist.set_ha('left')
                    artist.set_va('top')
                    artist.set_linespacing(1.10)
            legend = ax.get_legend()
            if legend is not None:
                for label in legend.get_texts():
                    label.set_fontsize(max(11.5, label.get_fontsize()))
                legend.get_title().set_fontsize(max(12, legend.get_title().get_fontsize()))
        if fig._suptitle is not None:
            fig._suptitle.set_fontsize(max(20, fig._suptitle.get_fontsize()))
            fig._suptitle.set_y(0.985)
            fig._suptitle.set_va('top')
        for legend in fig.legends:
            for label in legend.get_texts():
                label.set_fontsize(max(11.5, label.get_fontsize()))
        # Repeat after layout: wrapping changes title height and required margins.
        for _ in range(3):
            fig.canvas.draw()
            renderer = fig.canvas.get_renderer()
            if fig._suptitle is not None:
                _wrap_to_width(fig._suptitle, renderer, fig.bbox.width * 0.94)
            for ax in fig.axes:
                if ax.get_visible():
                    _wrap_to_width(ax.title, renderer, ax.bbox.width * 0.96)
            bottom, top = 0.02, 0.98
            if fig._suptitle is not None:
                title_height = fig._suptitle.get_window_extent(renderer).height / fig.bbox.height
                top = min(top, 0.97 - title_height - 0.018)
            for legend in fig.legends:
                bounds = legend.get_window_extent(renderer)
                fraction = bounds.height / fig.bbox.height
                if bounds.y0 + bounds.height / 2 < fig.bbox.y0 + fig.bbox.height / 2:
                    bottom = max(bottom, fraction + 0.045)
                else:
                    top = min(top, 0.97 - fraction - 0.025)
            # Let tight_layout control GridSpec spacing instead of skipping grids
            # whose hspace/wspace was explicitly fixed by older plotting code.
            engine = fig.get_layout_engine()
            if engine is None or engine.__class__.__name__ != 'ConstrainedLayoutEngine':
                grids = set()
                for ax in fig.axes:
                    if hasattr(ax, 'get_subplotspec'):
                        spec = ax.get_subplotspec()
                        if spec is not None:
                            grids.add(spec.get_gridspec())
                for grid in grids:
                    grid.update(hspace=None, wspace=None)
            if engine is not None and engine.__class__.__name__ == 'ConstrainedLayoutEngine':
                engine.set(w_pad=0.12, h_pad=0.16, wspace=0.10, hspace=0.15)
            else:
                with warnings.catch_warnings():
                    warnings.filterwarnings('ignore', message='.*not compatible with tight_layout.*')
                    fig.tight_layout(pad=1.25, h_pad=2.1, w_pad=2.2, rect=(0.01, bottom, 0.99, top))
            fig.canvas.draw()
            renderer = fig.canvas.get_renderer()
            # A long statistics block must not extend below its allocated panel.
            required_growth = 1.0
            for ax in fig.axes:
                for artist in ax.texts:
                    # Reserve room inside axes for bar-end numbers and log-scale
                    # percentage annotations; do not move their data coordinates.
                    if artist.get_transform() is ax.transData and ax.axison:
                        bounds = artist.get_window_extent(renderer)
                        x0, x1 = ax.get_xlim()
                        y0, y1 = ax.get_ylim()
                        if x1 > x0 and bounds.x1 > ax.bbox.x1 + 1:
                            right = ax.transData.inverted().transform((bounds.x1 + 8, ax.bbox.y0))[0]
                            if np.isfinite(right):
                                ax.set_xlim(x0, right)
                        if y1 > y0 and bounds.y1 > ax.bbox.y1 + 1:
                            upper = ax.transData.inverted().transform((ax.bbox.x0, bounds.y1 + 8))[1]
                            if np.isfinite(upper):
                                ax.set_ylim(y0, upper)
                    if not ax.axison and artist.get_text().count('\n') >= 8:
                        bounds = artist.get_window_extent(renderer)
                        required_growth = max(required_growth, bounds.height / max(ax.bbox.height * 0.94, 1))
                # Numeric ticks can use fewer positions while preserving their scale.
                labels = [t for t in ax.get_xticklabels() if t.get_visible() and t.get_text()]
                boxes = [t.get_window_extent(renderer) for t in labels]
                overlaps = any(a.overlaps(b) for a, b in zip(boxes, boxes[1:]))
                if overlaps and not ax.xaxis.get_units() and len(labels) > 5:
                    if all(t.get_rotation() == 0 for t in labels):
                        locator = ax.xaxis.get_major_locator()
                        if locator.__class__.__name__ in {'AutoLocator', 'MaxNLocator'}:
                            ax.xaxis.set_major_locator(MaxNLocator(nbins=4))
            if required_growth > 1.01:
                width, height = fig.get_size_inches()
                fig.set_size_inches(width, height * min(required_growth * 1.05, 1.6), forward=True)
        # Avoid collisions between annotation text and an axes legend.
        fig.canvas.draw()
        renderer = fig.canvas.get_renderer()
        for ax in fig.axes:
            legend = ax.get_legend()
            if legend is None:
                continue
            legend_box = legend.get_window_extent(renderer)
            if any(t.get_visible() and t.get_text() and legend_box.overlaps(t.get_window_extent(renderer))
                   for t in ax.texts):
                if hasattr(legend, 'set_loc'):
                    for location in ('upper left', 'lower right', 'lower left'):
                        legend.set_loc(location)
                        fig.canvas.draw()
                        legend_box = legend.get_window_extent(renderer)
                        if not any(t.get_visible() and t.get_text() and legend_box.overlaps(t.get_window_extent(renderer))
                                   for t in ax.texts):
                            break
        fig.canvas.draw()
        return fig
    finally:
        fig._readable_preparing = False


def _cache_figure_pdf(fig, source_name=None):
    """Snapshot vector PDFs in memory so closed figures survive until the last cell."""
    global _pdf_next_number
    if not fig.axes or not any(ax.get_visible() for ax in fig.axes):
        return
    key = getattr(fig, '_readable_pdf_key', None)
    if key is None or not str(key).startswith(NOTEBOOK_SLUG + '__'):
        key = f'{NOTEBOOK_SLUG}__{_pdf_next_number:03d}'
        _pdf_next_number += 1
        fig._readable_pdf_key = key
    previous = _pdf_figure_cache.get(key, {})
    name = previous.get('name')
    if source_name is not None and isinstance(source_name, (str, os.PathLike)):
        name = Path(source_name).stem
    if not name:
        title = getattr(fig, '_readable_export_title', '')
        if not title:
            title = getattr(fig, '_readable_export_axes_title', '')
        if not title:
            title = fig._suptitle.get_text() if fig._suptitle is not None else ''
        if not title:
            title = next((ax.get_title() for ax in fig.axes if ax.get_title()), 'figure')
        name = title
    name = re.sub(r'[^A-Za-z0-9._-]+', '_', name).strip('._')[:120] or 'figure'
    buffer = BytesIO()
    fig.savefig(buffer, format='pdf', bbox_inches='tight',
                pad_inches=getattr(fig, '_readable_pdf_pad', 0.25),
                metadata={'Creator': 'Matplotlib', 'Title': name})
    _pdf_figure_cache[key] = {
        'name': name, 'pdf': buffer.getvalue(),
        'export_name': getattr(fig, '_readable_export_name', None),
    }


def save_readable_figure(fig, filename, *args, **kwargs):
    prepare_readable_figure(fig)
    kwargs.setdefault('bbox_inches', 'tight')
    kwargs.setdefault('pad_inches', 0.25)
    result = fig.savefig(filename, *args, **kwargs)
    _cache_figure_pdf(fig, source_name=filename)
    return result


def save_readable_current_figure(filename, *args, **kwargs):
    return save_readable_figure(plt.gcf(), filename, *args, **kwargs)


def show_readable_figures(*args, **kwargs):
    for manager in list(Gcf.get_all_fig_managers()):
        fig = manager.canvas.figure
        prepare_readable_figure(fig)
        _cache_figure_pdf(fig)
    return plt.show(*args, **kwargs)


def close_readable_figures(fig=None):
    if fig == 'all':
        figures = [m.canvas.figure for m in Gcf.get_all_fig_managers()]
    elif fig is None:
        manager = Gcf.get_active()
        figures = [manager.canvas.figure] if manager is not None else []
    elif hasattr(fig, 'savefig'):
        figures = [fig]
    else:
        manager = Gcf.get_fig_manager(fig) if isinstance(fig, int) else None
        figures = [manager.canvas.figure] if manager is not None else []
    for figure in figures:
        prepare_readable_figure(figure)
        _cache_figure_pdf(figure)
    return plt.close(fig)


def export_notebook_figures():
    """Write every figure created during this run, including already-closed figures."""
    for manager in list(Gcf.get_all_fig_managers()):
        fig = manager.canvas.figure
        prepare_readable_figure(fig)
        _cache_figure_pdf(fig)
    if not _pdf_figure_cache:
        raise RuntimeError('No figures captured. Run the setup and plotting cells first.')
    FIGURES_PDF_DIR.mkdir(parents=True, exist_ok=True)
    saved = []
    for key, record in _pdf_figure_cache.items():
        filename = record.get('export_name') or f'{key}__{record["name"]}.pdf'
        path = FIGURES_PDF_DIR / filename
        path.write_bytes(record['pdf'])
        saved.append(path)
    print(f'Saved {len(saved)} PDF figures to: {FIGURES_PDF_DIR}')
    for path in saved:
        print('  ' + path.name)
    return saved

print('Readable plotting enabled. Final PDF destination:', FIGURES_PDF_DIR)


In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from joblib import load, dump
import os
import tables
import pathlib
from IPython.display import display
from matplotlib.colors import Normalize
from matplotlib import cm
from matplotlib.colors import LinearSegmentedColormap
from scipy.stats import gaussian_kde
from sklearn.metrics import r2_score


import warnings
warnings.filterwarnings("ignore")
    
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import pearsonr

### Datasets

In [ ]:
corrs = load('xgb_res/correlations.joblib')

In [ ]:
proteomics = load('xgb_res/proteomics_matched.joblib')

In [ ]:
tables.file._open_files.close_all()
p = pathlib.Path(r"xgb_res/xgb_all_results.h5")
print("exists:", p.exists(), "size:", p.stat().st_size, "bytes")
# Load all data from the HDF5 file
h5_file = 'xgb_res/xgb_all_results.h5'
eps = 1e-8

with pd.HDFStore(h5_file) as store:
    keys = store.keys()
    df_list = [store[key] for key in keys]
    combined_df = pd.concat(df_list, ignore_index=True)
print("Combined DataFrame shape:", combined_df.shape)


In [ ]:
combined_df

### Draft metrics plots

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score
from scipy.stats import pearsonr
from numpy.linalg import norm

# -----------------------------
# Helpers
# -----------------------------
def pearson_correlation(y_true, y_pred):
    if len(y_true) <= 1:
        return np.nan
    if np.std(y_true) == 0 or np.std(y_pred) == 0:
        return np.nan
    try:
        return pearsonr(y_true, y_pred)[0]
    except Exception:
        return np.nan

def cosine_similarity_metric(y_true, y_pred):
    if len(y_true) == 0:
        return np.nan
    denom = norm(y_true) * norm(y_pred)
    if denom == 0:
        return np.nan
    return np.dot(y_true, y_pred) / denom

def mse_metric(y_true, y_pred):
    if len(y_true) == 0:
        return np.nan
    return np.mean((y_true - y_pred) ** 2)

def _safe_r2(y_true, y_pred):
    if len(y_true) <= 1:
        return np.nan
    try:
        return r2_score(y_true, y_pred)
    except Exception:
        return np.nan

def _clean_pair(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = ~(np.isnan(y_true) | np.isnan(y_pred))
    return y_true[mask], y_pred[mask]

def _compute_metric_row(y_true, y_pred):
    y_true, y_pred = _clean_pair(y_true, y_pred)
    if len(y_true) <= 1:
        return None
    return {
        "pearson": pearson_correlation(y_true, y_pred),
        "mse": mse_metric(y_true, y_pred),
        "cosine": cosine_similarity_metric(y_true, y_pred),
        "r2": _safe_r2(y_true, y_pred),
    }

def _add_panel_label(ax, label):
    ax.text(
        0.01, 0.99, label,
        transform=ax.transAxes,
        fontsize=18.8,
        fontweight="bold",
        va="top",
        ha="left",
        bbox=dict(boxstyle="round,pad=0.2", facecolor="white", edgecolor="black", alpha=0.9),
    )

def _get_group_colors(group):
    if group == "proteins":
        return {
            "pearson": "#F5B041",
            "mse": "#E74C3C",
            "cosine": "#5DADE2",
            "r2": "#AF7AC5",
        }
    elif group == "samples":
        return {
            "pearson": "#F8C471",
            "mse": "#EC7063",
            "cosine": "#85C1E9",
            "r2": "#C39BD3",
        }
    else:
        raise ValueError("group must be either 'samples' or 'proteins'")

def _plot_metric_hist(ax, values, title, xlabel, color, fmt=".3f", panel_label=None):
    values = pd.Series(values).dropna()

    ax.set_facecolor("#FFFFFF")
    if len(values) > 0:
        ax.hist(values, bins=30, color=color, edgecolor="black", alpha=0.8)
        mean_val = values.mean()
        if np.isfinite(mean_val):
            ax.axvline(mean_val, color="red", linestyle="--", linewidth=2)
    else:
        ax.text(0.5, 0.5, "No valid values", ha="center", va="center", transform=ax.transAxes)

    ax.set_xlabel(xlabel, fontsize=14.5)
    ax.set_ylabel("Count", fontsize=14.5)
    _remember_axes_name(ax, title)

    if panel_label is not None:
        _add_panel_label(ax, panel_label)

def _build_summary_text(df, title_label, count_label):
    n_items = len(df)

    summary_text = f"""{title_label} Summary
{'='*28}
{count_label} (n={n_items})

Pearson:
  Mean: {df['pearson'].mean():.4f} +/- {df['pearson'].std():.4f}
  Min:  {df['pearson'].min():.4f}
  Max:  {df['pearson'].max():.4f}

MSE:
  Mean: {df['mse'].mean():.4f} +/- {df['mse'].std():.4f}
  Min:  {df['mse'].min():.4f}
  Max:  {df['mse'].max():.4f}

Cosine Similarity:
  Mean: {df['cosine'].mean():.4f} +/- {df['cosine'].std():.4f}
  Min:  {df['cosine'].min():.4f}
  Max:  {df['cosine'].max():.4f}

R²:
  Mean: {df['r2'].mean():.4f} +/- {df['r2'].std():.4f}
  Min:  {df['r2'].min():.4f}
  Max:  {df['r2'].max():.4f}
"""
    return summary_text

# -----------------------------
# Build detailed tables from combined_df
# -----------------------------
def compute_detailed_tables_from_combined(
    combined_df,
    sample_col="LOO sample",
    protein_col="protein",
    true_col="true",
    pred_col="predicted",
):
    required_cols = [sample_col, protein_col, true_col, pred_col]
    missing = [c for c in required_cols if c not in combined_df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    df = combined_df[[sample_col, protein_col, true_col, pred_col]].copy()
    df[true_col] = pd.to_numeric(df[true_col], errors="coerce")
    df[pred_col] = pd.to_numeric(df[pred_col], errors="coerce")

    # Per-sample metrics (across proteins)
    per_sample = {
        "sample": [],
        "pearson": [],
        "mse": [],
        "cosine": [],
        "r2": [],
    }

    for sample, g in df.groupby(sample_col):
        row = _compute_metric_row(g[true_col].values, g[pred_col].values)
        if row is not None:
            per_sample["sample"].append(sample)
            per_sample["pearson"].append(row["pearson"])
            per_sample["mse"].append(row["mse"])
            per_sample["cosine"].append(row["cosine"])
            per_sample["r2"].append(row["r2"])

    per_sample_df = pd.DataFrame(per_sample).sort_values("pearson", ascending=False, na_position="last")

    # Per-protein metrics (across samples)
    per_protein = {
        "protein": [],
        "pearson": [],
        "mse": [],
        "cosine": [],
        "r2": [],
    }

    for protein, g in df.groupby(protein_col):
        row = _compute_metric_row(g[true_col].values, g[pred_col].values)
        if row is not None:
            per_protein["protein"].append(protein)
            per_protein["pearson"].append(row["pearson"])
            per_protein["mse"].append(row["mse"])
            per_protein["cosine"].append(row["cosine"])
            per_protein["r2"].append(row["r2"])

    per_protein_df = pd.DataFrame(per_protein).sort_values("pearson", ascending=False, na_position="last")

    best_10_samples = per_sample_df.sort_values("pearson", ascending=False, na_position="last").head(10)
    worst_10_samples = per_sample_df.sort_values("pearson", ascending=True, na_position="last").head(10)

    best_10_proteins = per_protein_df.sort_values("pearson", ascending=False, na_position="last").head(10)
    worst_10_proteins = per_protein_df.sort_values("pearson", ascending=True, na_position="last").head(10)

    return (
        per_sample_df,
        per_protein_df,
        best_10_samples,
        worst_10_samples,
        best_10_proteins,
        worst_10_proteins,
    )

# -----------------------------
# Mean metrics from combined_df
# -----------------------------
def compute_mean_metrics_from_combined(
    combined_df,
    group="samples",
    sample_col="LOO sample",
    protein_col="protein",
    true_col="true",
    pred_col="predicted",
):
    if group not in {"samples", "proteins"}:
        raise ValueError("group must be either 'samples' or 'proteins'")

    (
        per_sample_df,
        per_protein_df,
        _,
        _,
        _,
        _,
    ) = compute_detailed_tables_from_combined(
        combined_df,
        sample_col=sample_col,
        protein_col=protein_col,
        true_col=true_col,
        pred_col=pred_col,
    )

    df = per_sample_df if group == "samples" else per_protein_df

    out = {
        "Pearson": np.nanmean(df["pearson"]) if len(df) else np.nan,
        "MSE": np.nanmean(df["mse"]) if len(df) else np.nan,
        "Cosine Similarity": np.nanmean(df["cosine"]) if len(df) else np.nan,
        "R2": np.nanmean(df["r2"]) if len(df) else np.nan,
    }
    out["Cosine"] = out["Cosine Similarity"]
    return out

# -----------------------------
# Overall bar plots
# -----------------------------
def plot_overall_bar(mean_metrics, out_path=None, label="XGBoost", group="proteins", title=None):
    plt.style.use("seaborn-v0_8-whitegrid")
    colors = _get_group_colors(group)

    fig, axes = plt.subplots(2, 2, figsize=readable_figsize((12, 8)))
    fig.patch.set_facecolor("#F8F9FA")

    if title is not None:
        _remember_figure_name(fig, title)

    plots = [
        ("Pearson Correlation", "PCC", mean_metrics.get("Pearson", np.nan), colors["pearson"], "a)"),
        ("Mean Squared Error", "MSE", mean_metrics.get("MSE", np.nan), colors["mse"], "b)"),
        ("Cosine Similarity", "Cosine Similarity", mean_metrics.get("Cosine Similarity", mean_metrics.get("Cosine", np.nan)), colors["cosine"], "c)"),
        ("R² Score", "R²", mean_metrics.get("R2", np.nan), colors["r2"], "d)"),
    ]

    for ax, (plot_title, ylabel, value, color, panel_label) in zip(axes.flatten(), plots):
        ax.set_facecolor("#FFFFFF")

        if np.isfinite(value):
            ax.bar(
                [label], [value],
                color=color,
                width=0.6,
                edgecolor="#2C3E50",
                linewidth=2,
                alpha=0.9
            )

            if value >= 0:
                ax.set_ylim(0, value * 1.35 if value != 0 else 1)
            else:
                ax.set_ylim(value * 1.35, abs(value) * 0.35)
        else:
            ax.text(0.5, 0.5, "No valid values", ha="center", va="center", transform=ax.transAxes)

        _remember_axes_name(ax, plot_title)
        ax.set_ylabel(ylabel, fontsize=15.9, fontweight="bold", color="#2C3E50")
        ax.tick_params(axis="x", labelsize=15.9, colors="#2C3E50")
        ax.tick_params(axis="y", labelsize=14.5, colors="#7F8C8D")
        ax.yaxis.grid(True, linestyle="--", alpha=0.3)
        ax.set_axisbelow(True)

        for spine in ax.spines.values():
            spine.set_color("#BDC3C7")
            spine.set_linewidth(1.2)

        _add_panel_label(ax, panel_label)

    if title is not None:
        plt.tight_layout(rect=[0, 0, 1, 0.97])
    else:
        plt.tight_layout()

    if out_path is not None:
        save_readable_current_figure(out_path, dpi=200, bbox_inches="tight", facecolor=fig.get_facecolor())
        print(f"Saved: {out_path}")

    show_readable_figures()

# -----------------------------
# Detailed 6-panel plots
# -----------------------------
def plot_detailed(per_sample_df=None, per_protein_df=None, group="samples", out_path=None, title=None):
    if group not in {"samples", "proteins"}:
        raise ValueError("group must be either 'samples' or 'proteins'")

    colors = _get_group_colors(group)

    fig = plt.figure(figsize=readable_figsize((16, 16)))
    gs = fig.add_gridspec(3, 2, hspace=0.42, wspace=0.42)

    if title is not None:
        _remember_figure_name(fig, title)

    if group == "samples":
        if per_sample_df is None:
            raise ValueError("per_sample_df is required when group='samples'")

        df = per_sample_df.copy()

        ax1 = fig.add_subplot(gs[0, 0])
        _plot_metric_hist(ax1, df["pearson"], "Per-Sample PCC", "Pearson Correlation", colors["pearson"], panel_label="a)")

        ax2 = fig.add_subplot(gs[0, 1])
        _plot_metric_hist(ax2, df["mse"], "Per-Sample MSE", "MSE", colors["mse"], fmt=".4f", panel_label="b)")

        ax3 = fig.add_subplot(gs[1, 0])
        _plot_metric_hist(ax3, df["cosine"], "Per-Sample Cosine Similarity", "Cosine Similarity", colors["cosine"], panel_label="c)")

        ax4 = fig.add_subplot(gs[1, 1])
        _plot_metric_hist(ax4, df["r2"], "Per-Sample R²", "R² Score", colors["r2"], panel_label="d)")

        ax5 = fig.add_subplot(gs[2, 0])
        best_10 = df.sort_values("pearson", ascending=False, na_position="last").head(10).sort_values("pearson", ascending=True)
        worst_10 = df.sort_values("pearson", ascending=True, na_position="last").head(10).sort_values("pearson", ascending=False)
        combined = pd.concat([best_10, worst_10], axis=0)

        bar_colors = (["#58D68D"] * len(best_10)) + (["#E74C3C"] * len(worst_10))
        y_pos = np.arange(len(combined))

        ax5.barh(y_pos, combined["pearson"].values, color=bar_colors, edgecolor="black", alpha=0.8)
        ax5.set_yticks(y_pos)
        ax5.set_yticklabels(combined["sample"].values, fontsize=11.6)
        ax5.axvline(0, color="red", linestyle="-", linewidth=1)
        ax5.set_xlabel("PCC", fontsize=14.5)
        _remember_axes_name(ax5, 'Best 10 / Worst 10 Samples by PCC')
        _add_panel_label(ax5, "e)")

        ax6 = fig.add_subplot(gs[2, 1])
        ax6.axis("off")
        _remember_axes_name(ax6, 'Summary Statistics')
        _add_panel_label(ax6, "f)")

        summary_text = _build_summary_text(
            df=df,
            title_label="Per-Sample",
            count_label="Samples",
        )

        ax6.text(
            0.5, 0.92, summary_text,
            transform=ax6.transAxes,
            fontsize=13.0,
            va="top",
            ha="center",
            multialignment="left",
            fontfamily="monospace",
            bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.45),
        )

    else:
        if per_protein_df is None:
            raise ValueError("per_protein_df is required when group='proteins'")

        df = per_protein_df.copy()

        ax1 = fig.add_subplot(gs[0, 0])
        _plot_metric_hist(ax1, df["pearson"], "Per-Protein PCC", "Pearson Correlation", colors["pearson"], panel_label="a)")

        ax2 = fig.add_subplot(gs[0, 1])
        _plot_metric_hist(ax2, df["mse"], "Per-Protein MSE", "MSE", colors["mse"], fmt=".4f", panel_label="b)")

        ax3 = fig.add_subplot(gs[1, 0])
        _plot_metric_hist(ax3, df["cosine"], "Per-Protein Cosine Similarity", "Cosine Similarity", colors["cosine"], panel_label="c)")

        ax4 = fig.add_subplot(gs[1, 1])
        _plot_metric_hist(ax4, df["r2"], "Per-Protein R²", "R² Score", colors["r2"], panel_label="d)")

        ax5 = fig.add_subplot(gs[2, 0])
        best_10 = df.sort_values("pearson", ascending=False, na_position="last").head(10).sort_values("pearson", ascending=True)
        worst_10 = df.sort_values("pearson", ascending=True, na_position="last").head(10).sort_values("pearson", ascending=False)
        combined = pd.concat([best_10, worst_10], axis=0)

        bar_colors = (["#58D68D"] * len(best_10)) + (["#E74C3C"] * len(worst_10))
        y_pos = np.arange(len(combined))

        ax5.barh(y_pos, combined["pearson"].values, color=bar_colors, edgecolor="black", alpha=0.8)
        ax5.set_yticks(y_pos)
        ax5.set_yticklabels(combined["protein"].values, fontsize=11.6)
        ax5.axvline(0, color="red", linestyle="-", linewidth=1)
        ax5.set_xlabel("PCC", fontsize=14.5)
        _remember_axes_name(ax5, 'Best 10 / Worst 10 Proteins by PCC')
        _add_panel_label(ax5, "e)")

        ax6 = fig.add_subplot(gs[2, 1])
        ax6.axis("off")
        _remember_axes_name(ax6, 'Summary Statistics')
        _add_panel_label(ax6, "f)")

        summary_text = _build_summary_text(
            df=df,
            title_label="Per-Protein",
            count_label="Proteins",
        )

        ax6.text(
            0.5, 0.92, summary_text,
            transform=ax6.transAxes,
            fontsize=13.0,
            va="top",
            ha="center",
            multialignment="left",
            fontfamily="monospace",
            bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.45),
        )

    if title is not None:
        plt.tight_layout(rect=[0, 0, 1, 0.96])
    else:
        plt.tight_layout()

    if out_path is not None:
        save_readable_current_figure(out_path, dpi=150, bbox_inches="tight")
        print(f"Saved: {out_path}")

    show_readable_figures()

# -----------------------------
# Run everything from combined_df
# -----------------------------
(
    per_sample_df,
    per_protein_df,
    best_10_samples,
    worst_10_samples,
    best_10_proteins,
    worst_10_proteins,
) = compute_detailed_tables_from_combined(combined_df)

mean_metrics_samples = compute_mean_metrics_from_combined(combined_df, group="samples")
mean_metrics_proteins = compute_mean_metrics_from_combined(combined_df, group="proteins")

print("Per-sample mean metrics:")
print(mean_metrics_samples)
print("\nPer-protein mean metrics:")
print(mean_metrics_proteins)

# Overall bars
plot_overall_bar(mean_metrics_samples, group="samples", label="XGBoost")
plot_overall_bar(mean_metrics_proteins, group="proteins", label="XGBoost")

# Detailed panels
plot_detailed(per_sample_df=per_sample_df, group="samples")
plot_detailed(per_protein_df=per_protein_df, group="proteins")

In [ ]:
# Non-finite overview (NaN / +Inf / -Inf) for combined_df

df = combined_df  # already loaded above
n_rows, n_cols = df.shape

# split columns
numeric_cols = df.select_dtypes(include=[np.number]).columns
non_numeric_cols = df.columns.difference(numeric_cols)

# per-column counts
nan_counts = df.isna().sum()

# ±Inf only make sense for numeric columns
if len(numeric_cols) > 0:
    posinf_counts_num = pd.Series(
        np.isposinf(df[numeric_cols].to_numpy()).sum(axis=0), index=numeric_cols, dtype="int64"
    )
    neginf_counts_num = pd.Series(
        np.isneginf(df[numeric_cols].to_numpy()).sum(axis=0), index=numeric_cols, dtype="int64"
    )
else:
    posinf_counts_num = pd.Series(dtype="int64")
    neginf_counts_num = pd.Series(dtype="int64")

# align to all columns (0 for non-numeric)
posinf_counts = pd.Series(0, index=df.columns, dtype="int64")
neginf_counts = pd.Series(0, index=df.columns, dtype="int64")
posinf_counts.loc[posinf_counts_num.index] = posinf_counts_num
neginf_counts.loc[neginf_counts_num.index] = neginf_counts_num

summary = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "n_nan": nan_counts.astype("int64"),
    "n_posinf": posinf_counts,
    "n_neginf": neginf_counts,
})
summary["n_inf"] = summary["n_posinf"] + summary["n_neginf"]
summary["n_nonfinite"] = summary["n_nan"] + summary["n_inf"]
summary["pct_nonfinite"] = (summary["n_nonfinite"] / max(n_rows, 1)) * 100

# row-level stats
nan_any_row = df.isna().any(axis=1).to_numpy()

if len(numeric_cols) > 0:
    inf_mask = np.isinf(df[numeric_cols].to_numpy())
    inf_any_row = inf_mask.any(axis=1)
else:
    inf_any_row = np.zeros(n_rows, dtype=bool)

nonfinite_any_row = nan_any_row | inf_any_row

print(f"Rows: {n_rows}  |  Columns: {n_cols} (numeric: {len(numeric_cols)}, non-numeric: {len(non_numeric_cols)})")
print(f"Rows with any NaN: {int(nan_any_row.sum())}")
print(f"Rows with any ±Inf (numeric only): {int(inf_any_row.sum())}")
print(f"Rows with any non-finite (NaN or ±Inf): {int(nonfinite_any_row.sum())}\n")

# show columns with most issues first
summary_sorted = summary.sort_values("n_nonfinite", ascending=False)

# pretty display
display(
    summary_sorted.style
        .format({"pct_nonfinite": "{:.2f}%", "n_nan": "{:d}", "n_posinf": "{:d}", "n_neginf": "{:d}", "n_inf": "{:d}", "n_nonfinite": "{:d}"})
        .bar(subset=["n_nonfinite"], color="#FFA07A")
        .bar(subset=["n_nan"], color="#FFDAB9")
        .bar(subset=["n_inf"], color="#FFC0CB")
)

# optional: quick list of the top 10 worst columns
print("\nTop 10 columns by non-finite values:")
display(summary_sorted.head(10)[["dtype","n_nonfinite","n_nan","n_inf","pct_nonfinite"]])


In [ ]:

df = combined_df.copy()

# choose the sample column
sample_col = 'LOO sample' if 'LOO sample' in df.columns else 'sample'

# keep only numeric/finite true & predicted
df['true'] = pd.to_numeric(df['true'], errors='coerce')
df['predicted'] = pd.to_numeric(df['predicted'], errors='coerce')
df = df.replace([np.inf, -np.inf], np.nan)

# mask of rows with finite true & predicted
row_ok = df[['true', 'predicted']].notna().all(axis=1)

# A) proteins present after dropping all non-finite rows (≥1 valid row remains)
n_proteins_total = df['protein'].nunique()
n_proteins_after_drop = df.loc[row_ok, 'protein'].nunique()

print(f"Unique proteins (total): {n_proteins_total}")
print(f"Unique proteins with at least one valid row after removing NaNs/±Inf: {n_proteins_after_drop}")

# B) proteins complete across *all* their rows (no NaN/±Inf anywhere)
protein_complete = (
    df.groupby('protein')[['true', 'predicted']]
      .apply(lambda g: g.notna().all().all())
)
n_proteins_complete = int(protein_complete.sum())
print(f"Proteins complete across all rows (no NaN/±Inf in true/predicted): {n_proteins_complete}")

# C) per-sample: how many proteins remain after removing NaNs/±Inf (counts of valid rows)
per_sample_counts = (
    df.groupby(sample_col)
      .apply(lambda g: g[['true','predicted']].notna().all(axis=1).sum())
      .reset_index(name='n_proteins_remaining')
)

print("\nPer-sample proteins remaining (first 10 rows):")
display(per_sample_counts.head(10))

print("\nPer-sample summary:")
print(
    per_sample_counts['n_proteins_remaining']
      .agg(['min','median','max'])
      .rename({'min':'min', 'median':'median', 'max':'max'})
      .to_string()
)


In [ ]:
# Clean combined_df by removing entire proteins that contain any NaN/±Inf in true/predicted
df = combined_df.copy()

# 1) Coerce to numeric and turn ±Inf into NaN
df["true"] = pd.to_numeric(df["true"], errors="coerce")
df["predicted"] = pd.to_numeric(df["predicted"], errors="coerce")
df = df.replace([np.inf, -np.inf], np.nan)

# 2) Which proteins are COMPLETE (no NaN in any of their rows for true/predicted)?
protein_complete = (
    df.groupby("protein")[["true", "predicted"]]
      .apply(lambda g: g.notna().all().all())
)
kept_proteins = protein_complete[protein_complete].index

# 3) Filter to the cleaned dataset
combined_df_clean = df[df["protein"].isin(kept_proteins)].copy()

# 4) New NaN/Inf overview (should be zero for true/predicted)
n_rows_before = len(df)
n_rows_after  = len(combined_df_clean)
n_proteins_before = df["protein"].nunique()
n_proteins_after  = combined_df_clean["protein"].nunique()

nan_true_before = df["true"].isna().sum()
nan_pred_before = df["predicted"].isna().sum()
nan_true_after  = combined_df_clean["true"].isna().sum()
nan_pred_after  = combined_df_clean["predicted"].isna().sum()

print(f"Rows before:  {n_rows_before:,}   | proteins before:  {n_proteins_before:,}")
print(f"Rows after:   {n_rows_after:,}    | proteins after:   {n_proteins_after:,}")
print(f"Proteins removed: {n_proteins_before - n_proteins_after:,}")

print("\nNaN check (true/predicted):")
print(f"  Before  -> true: {nan_true_before:,}  | predicted: {nan_pred_before:,}")
print(f"  After   -> true: {nan_true_after:,}   | predicted: {nan_pred_after:,}")

# Optional: quick all-column non-finite check on the cleaned frame
numeric_cols = combined_df_clean.select_dtypes(include=[np.number]).columns
rows_with_any_nan = combined_df_clean.isna().any(axis=1).sum()
rows_with_any_inf = np.isinf(combined_df_clean[numeric_cols].to_numpy()).any(axis=1).sum() if len(numeric_cols) else 0
print(f"\nRows with any NaN (any column) in cleaned df: {rows_with_any_nan:,}")
print(f"Rows with any ±Inf (numeric cols) in cleaned df: {rows_with_any_inf:,}")

print("\nFinal cleaned shape:", combined_df_clean.shape)

combined_df = combined_df_clean


In [ ]:
# scTranslator-style per-sample min–max normalization for a long DataFrame
# expected columns in `combined_df`: ["protein", "LOO sample", "true", "predicted"]

EPS = 1e-8  # lower bound in the target scale

# --- sanity checks
required_cols = {"protein", "LOO sample", "true", "predicted"}
missing = required_cols - set(combined_df.columns)
if missing:
    raise ValueError(f"combined_df is missing required columns: {sorted(missing)}")

df = combined_df.copy()

# ensure numeric
df["true"] = pd.to_numeric(df["true"], errors="coerce")
df["predicted"] = pd.to_numeric(df["predicted"], errors="coerce")

# compute truth-based min/max per sample (LOO sample)
g = df.groupby("LOO sample", observed=True)["true"]
mins = g.min().rename("tmin")
maxs = g.max().rename("tmax")
mm = pd.concat([mins, maxs], axis=1)

# merge back
df = df.merge(mm, left_on="LOO sample", right_index=True, how="left")

# scale function using truth min/max for the sample
denom = (df["tmax"] - df["tmin"]).to_numpy()
denom = np.where(denom == 0, np.nan, denom)  # mark degenerate samples

def scale_series(x, tmin, denom):
    out = EPS + (x.to_numpy() - tmin.to_numpy()) / denom * (1.0 - EPS)
    # if denom is nan (constant truth vector), set all to EPS
    out = np.where(np.isnan(out), EPS, out)
    return pd.Series(out, index=x.index, dtype="float64")

df["true_mm"] = scale_series(df["true"], df["tmin"], denom)
df["pred_mm"] = scale_series(df["predicted"], df["tmin"], denom)

# clip to be extra-safe against tiny floating drift
df["true_mm"]  = df["true_mm"].clip(lower=EPS, upper=1.0)
df["pred_mm"]  = df["pred_mm"].clip(lower=EPS, upper=1.0)

# quick validation
print("Ranges after scaling:")
print("  true_mm :", float(df["true_mm"].min()), "→", float(df["true_mm"].max()))
print("  pred_mm :", float(df["pred_mm"].min()), "→", float(df["pred_mm"].max()))

# OPTIONAL: per-protein Pearson r on the min–max scale
# (pivot to samples × proteins, then correlate column-wise)
want_metrics = True
if want_metrics:
    t_mat = df.pivot(index="LOO sample", columns="protein", values="true_mm")
    p_mat = df.pivot(index="LOO sample", columns="protein", values="pred_mm")

    # align columns just in case
    common = t_mat.columns.intersection(p_mat.columns)
    t_mat = t_mat[common]
    p_mat = p_mat[common]

    # vectorized Pearson r per protein
    t = t_mat.to_numpy(dtype=float)
    p = p_mat.to_numpy(dtype=float)

    # subtract means along samples
    t_z = t - np.nanmean(t, axis=0, keepdims=True)
    p_z = p - np.nanmean(p, axis=0, keepdims=True)

    num = np.nansum(t_z * p_z, axis=0)
    den = np.sqrt(np.nansum(t_z**2, axis=0) * np.nansum(p_z**2, axis=0))
    r = np.divide(num, den, out=np.full_like(num, np.nan, dtype=float), where=den!=0)

    per_protein_corr = pd.Series(r, index=common, name="pearson_r").sort_values(ascending=False)
    print("\nPer-protein Pearson r (first 10):")
    print(per_protein_corr.head(10))

# df now contains normalized columns:
#   - 'true_mm' : truth scaled to [1e-8, 1] per sample using truth min/max
#   - 'pred_mm' : predictions scaled with the SAME per-sample truth min/max
# You can use these to compare fairly with scTranslator outputs.


In [ ]:

# Compute absolute and squared errors
combined_df['abs_error'] = (combined_df['true'] - combined_df['predicted']).abs()
combined_df['squared_error'] = (combined_df['true'] - combined_df['predicted'])**2
combined_df['rel_error']     = combined_df['abs_error'] / (combined_df['true'].abs() + eps)
# Add overall Pearson and cosine similarity as columns
corr = combined_df['true'].corr(combined_df['predicted'])
cos  = combined_df['true'].mul(combined_df['predicted']).sum() / ((combined_df['true'].pow(2).sum()**0.5) * (combined_df['predicted'].pow(2).sum()**0.5) + eps)
combined_df['pearson_corr'] = corr
combined_df['cosine_similarity'] = cos

combined_df.head()


##  Protein-wise analysis


Compute per-protein metrics

In [ ]:

# Compute absolute and squared errors
combined_df['abs_error'] = (combined_df['true'] - combined_df['predicted']).abs()
combined_df['squared_error'] = (combined_df['true'] - combined_df['predicted'])**2
combined_df['rel_error']     = combined_df['abs_error'] / (combined_df['true'].abs() + eps)
# Add overall Pearson and cosine similarity as columns
corr = combined_df['true'].corr(combined_df['predicted'])
cos  = combined_df['true'].mul(combined_df['predicted']).sum() / ((combined_df['true'].pow(2).sum()**0.5) * (combined_df['predicted'].pow(2).sum()**0.5) + eps)
combined_df['pearson_corr'] = corr
combined_df['cosine_similarity'] = cos

combined_df.head()


In [ ]:


# Per-protein R² (across all LOO samples for that protein)
def _r2_group(g):
    y, yhat = g['true'].values, g['predicted'].values
    return r2_score(y, yhat) if (len(y) > 1 and np.var(y) > 0) else np.nan

r2_by_protein = (combined_df
                 .groupby('protein', dropna=False)
                 .apply(_r2_group)
                 .rename('r2')
                 .reset_index())

print("R² by protein — summary:")
print(r2_by_protein['r2'].describe())

# Histogram
plt.figure()
plt.hist(r2_by_protein['r2'].dropna().values, bins=30)
plt.xlabel('R² (per protein)')
plt.ylabel('Count')
_remember_figure_name(plt.gcf(), 'Distribution of R² across proteins')
plt.tight_layout()
show_readable_figures()

# Optional: extremes
print("\nTop 10 proteins by R²:")
display(r2_by_protein.sort_values('r2', ascending=False).head(10))
print("\nBottom 10 proteins by R²:")
display(r2_by_protein.sort_values('r2', ascending=True).head(10))



In [ ]:
r2_by_protein['r2'].median()

In [ ]:

# Per-sample R² (across proteins within each LOO sample)
def _r2_sample_group(g):
    y, yhat = g['true'].values, g['predicted'].values
    return r2_score(y, yhat) if (len(y) > 1 and np.var(y) > 0) else np.nan

r2_by_sample = (combined_df
                .groupby('LOO sample', dropna=False)
                .apply(_r2_sample_group)
                .rename('r2')
                .reset_index())

print("R² by sample — summary:")
print(r2_by_sample['r2'].describe())

# Histogram
plt.figure()
plt.hist(r2_by_sample['r2'].dropna().values, bins=30)
plt.xlabel('R² (per LOO sample)')
plt.ylabel('Count')
_remember_figure_name(plt.gcf(), 'Distribution of R² across LOO samples')
plt.tight_layout()
show_readable_figures()

# Optional: extremes
print("\nTop 10 LOO samples by R²:")
display(r2_by_sample.sort_values('r2', ascending=False).head(10))
print("\nBottom 10 LOO samples by R²:")
display(r2_by_sample.sort_values('r2', ascending=True).head(10))


In [ ]:
# pull clean squared errors
se = (pd.to_numeric(combined_df["squared_error"], errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
        .dropna())

# choose an x-axis cap (tweak if you like)
p95, p99, p999 = np.percentile(se, [95, 99, 99.9])
xmax = p99          # show up to the 99th percentile
tail_n = int((se > xmax).sum())

fig, ax = plt.subplots(figsize=readable_figsize((8, 5)), dpi=130)

# histogram only for values within the cap
data_in = se[se <= xmax]
bins = np.linspace(0, xmax, 60)
ax.hist(data_in, bins=bins)

# annotate mean/median (only if they fall within the shown range)
mean_val = se.mean()
med_val  = se.median()
if med_val <= xmax:
    ax.axvline(med_val, linestyle="--", linewidth=1.5, label=f"median = {med_val:.4g}")
else:
    ax.text(xmax, ax.get_ylim()[1]*0.90, f"median = {med_val:.4g} (off-scale)",
            ha="right", va="top", fontsize=13.0)

if mean_val <= xmax:
    ax.axvline(mean_val, linestyle="-", linewidth=1.5, label=f"mean = {mean_val:.4g}")
else:
    ax.text(xmax, ax.get_ylim()[1]*0.80, f"mean = {mean_val:.4g} (off-scale)",
            ha="right", va="top", fontsize=13.0)

# x-axis limits and a note about the hidden tail
ax.set_xlim(0, xmax)
if tail_n > 0:
    ax.text(xmax, ax.get_ylim()[1]*0.70, f">{xmax:.3g}: {tail_n} outliers",
            ha="right", va="top", fontsize=13.0)

_remember_axes_name(ax, 'Distribution of Squared Errors')
ax.set_xlabel("Squared error")
ax.set_ylabel("Count")
ax.legend(loc="upper right")
plt.tight_layout()
show_readable_figures()


In [ ]:
# Per-sample metrics from combined_df → samples_df (369 rows expected)
eps = 1e-12

combined_df = combined_df.copy()

def _cos_sim(a, b):
    na = np.linalg.norm(a); nb = np.linalg.norm(b)
    denom = na * nb
    return float(np.dot(a, b) / denom) if denom > 0 else np.nan

# the column holding sample IDs (handles the space in "LOO sample")
sample_col = 'LOO sample' if 'LOO sample' in combined_df.columns else 'sample'

samples_df = (
    combined_df.groupby(sample_col, sort=False)
      .apply(lambda g: pd.Series({
          'n_proteins': len(g),
          'mae': g['abs_error'].mean(),
          'mse': g['squared_error'].mean(),
          'rmse': np.sqrt(g['squared_error'].mean()),
          'cosine_similarity': _cos_sim(g['true'].to_numpy(), g['predicted'].to_numpy()),
          'pearson_corr': g['true'].corr(g['predicted'])
      }))
      .reset_index()
      .rename(columns={sample_col: 'sample'})
)

samples_df.head()


In [ ]:
samples_df.sort_values('mse', ascending=False).head(20)

In [ ]:
#  tissue counts from samples_df 

min_n = 18 # threshold for "low-representation" tissues

# Tissue is the suffix after the last underscore
samples_df = samples_df.copy()
samples_df["tissue"] = samples_df["sample"].astype(str).str.rsplit("_", n=1).str[-1]

# Count unique samples per tissue
tissue_counts = (
    samples_df.drop_duplicates("sample")
              .groupby("tissue")["sample"]
              .nunique()
              .sort_values()
)

low_tissues = tissue_counts[tissue_counts <= min_n]

print(f"Total tissues: {tissue_counts.size}")
print(f"Tissues with <= {min_n} samples:")
if low_tissues.empty:
    print("  (none)")
else:
    # pretty print as two columns
    print(low_tissues.to_string())


In [ ]:
#  filter combined_df by tissue representation 
n = 18  # set the threshold here

# Reuse tissue mapping from Cell 1 (samples_df already has a 'tissue' column)
tissue_sample_counts = (
    samples_df.drop_duplicates("sample")
              .groupby("tissue")["sample"]
              .nunique()
)

low_tissues = set(tissue_sample_counts[tissue_sample_counts < n].index)

# All samples that belong to low-representation tissues
low_tissue_samples = set(
    samples_df.loc[samples_df["tissue"].isin(low_tissues), "sample"].unique()
)

# combined_df may name the sample column as 'LOO_sample' or 'LOO sample'—we handle both.
sample_col = "LOO_sample" if "LOO_sample" in combined_df.columns else (
    "LOO sample" if "LOO sample" in combined_df.columns else "sample"
)

before = len(combined_df)
clean = combined_df.loc[~combined_df[sample_col].isin(low_tissue_samples)].copy()
after = len(clean)

print(f"Threshold n = {n}")
print(f"Low-representation tissues ({len(low_tissues)}): {sorted(low_tissues) if low_tissues else 'none'}")
print(f"Removed {before - after} rows; kept {after} rows.")


In [ ]:
def _cos_sim(a, b):
    na = np.linalg.norm(a); nb = np.linalg.norm(b)
    denom = na * nb
    return float(np.dot(a, b) / denom) if denom > 0 else np.nan

per_protein = (
    clean.groupby('protein', sort=False)
    .apply(lambda g: pd.Series({
        'n_samples': len(g),
        'mse': g['squared_error'].mean(),
        'cosine_similarity': _cos_sim(g['true'].to_numpy(), g['predicted'].to_numpy()),
        'pearson_corr': g['true'].corr(g['predicted'])
    }))
    .reset_index()
)

### Top 100 proteins by mean squared error-cosine similarity-pearson correlation

In [ ]:
# Per-protein ranking tables: MSE, cosine similarity, Pearson correlation


top100_mse     = per_protein.sort_values('mse', ascending=True).head(100)         # lower is better
top100_cosine  = per_protein.sort_values('cosine_similarity', ascending=False).head(100)
top100_pearson = per_protein.sort_values('pearson_corr', ascending=False).head(100)

print("Top 100 proteins by lowest MSE")
display(top100_mse)

print("\nTop 100 proteins by highest cosine similarity")
display(top100_cosine)

print("\nTop 100 proteins by highest Pearson correlation")
display(top100_pearson)

In [ ]:
# Intersection
set_mse  = set(top100_mse['protein'])
set_cos  = set(top100_cosine['protein'])
set_pear = set(top100_pearson['protein'])
inter_prots = sorted(set_mse & set_cos & set_pear)

print(f"Proteins in the intersection of all three top-100 lists: {len(inter_prots)}")

if len(inter_prots) == 0:
    print("No overlap across the three lists.")
else:
    # Add ranks for context
    pp_idx = per_protein.set_index('protein')
    ranked = pp_idx.assign(
        mse_rank=pp_idx['mse'].rank(method='min', ascending=True).astype('Int64'),
        cosine_rank=pp_idx['cosine_similarity'].rank(method='min', ascending=False).astype('Int64'),
        pearson_rank=pp_idx['pearson_corr'].rank(method='min', ascending=False).astype('Int64'),
    )

    intersection_table = (
        ranked.loc[inter_prots, ['n_samples', 'mse', 'mse_rank',
                                 'cosine_similarity', 'cosine_rank',
                                 'pearson_corr', 'pearson_rank']]
        .sort_values(['mse', 'cosine_rank', 'pearson_rank'])
        .reset_index()
        .rename(columns={'protein': 'protein'})
    )

    display(intersection_table)

### Error Distributions and thresholds 

#### means

In [ ]:
# Compute mean squared error for each protein across samples
protein_means = combined_df.groupby("protein")["squared_error"].mean().reset_index()
protein_means.rename(columns={"squared_error": "mean_squared_error"}, inplace=True)

# Plot the distribution
plt.figure(figsize=readable_figsize((10, 5)))
sns.histplot(protein_means["mean_squared_error"], bins=30, kde=True, color="steelblue")

_remember_figure_name(plt.gcf(), 'Distribution of Mean Squared Errors Across Proteins')
plt.xlabel("Mean Squared Error")
plt.ylabel("Number of Proteins")
plt.tight_layout()
show_readable_figures()

In [ ]:
# Compute mean squared error for each protein across samples
protein_means = clean.groupby("protein")["squared_error"].mean().reset_index()
protein_means.rename(columns={"squared_error": "mean_squared_error"}, inplace=True)

# Plot the distribution
plt.figure(figsize=readable_figsize((10, 5)))
sns.histplot(protein_means["mean_squared_error"], bins=30, kde=True, color="steelblue")

_remember_figure_name(plt.gcf(), 'Distribution of Mean Squared Errors Across Proteins')
plt.xlabel("Mean Squared Error")
plt.ylabel("Number of Proteins")
plt.tight_layout()
show_readable_figures()

In [ ]:
# MAE < 0.1
threshold = 0.1

# mask + subset for plotting
mask = protein_means["mean_squared_error"] < threshold
subset = protein_means.loc[mask, "mean_squared_error"]

print(f"Proteins with mean_squared_error < {threshold}: {subset.shape[0]}")

if subset.empty:
    print("No proteins below this threshold to plot.")
else:
    # --- histogram (seaborn) ---
    plt.figure(figsize=readable_figsize((10, 5)))
    sns.histplot(subset, bins=30, kde=True, color="steelblue")
    _remember_figure_name(plt.gcf(), f'Distribution of Mean Squared Errors (MSE < {threshold})')
    plt.xlabel("Mean Squared Error")
    plt.ylabel("Number of Proteins")
    plt.tight_layout()
    show_readable_figures()

    # --- table of proteins with MSE < threshold ---
    # If there's a 'protein' column, use it; otherwise, lift the index into a column.
    if "protein" in protein_means.columns:
        table = protein_means.loc[mask, ["protein", "mean_squared_error"]].copy()
    else:
        table = (
            protein_means.loc[mask, ["mean_squared_error"]]
            .reset_index()
            .rename(columns={"index": "protein"})
        )

    # sort for readability and format
    table = table.sort_values("mean_squared_error").reset_index(drop=True)
    table["mean_squared_error"] = table["mean_squared_error"].astype(float)

    # Display nicely in notebooks (falls back to print if display isn't available)
    try:
        from IPython.display import display
        print("\nTable: Proteins with MSE < {:.3f} (sorted ascending)".format(threshold))
        display(table.style.format({"mean_squared_error": "{:.6f}"}))
    except Exception:
        print("\nTable: Proteins with MSE < {:.3f} (sorted ascending)".format(threshold))
        print(table.to_string(index=False))


In [ ]:
# Cell: MAE < 0.2
threshold = 0.2
subset = protein_means.loc[protein_means["mean_squared_error"] < threshold, "mean_squared_error"]

print(f"Proteins with mean_squared_error < {threshold}: {subset.shape[0]}")

if subset.empty:
    print("No proteins below this threshold to plot.")
else:
    plt.figure(figsize=readable_figsize((10, 5)))
    sns.histplot(subset, bins=30, kde=True, color="steelblue")
    _remember_figure_name(plt.gcf(), 'Distribution of Mean Squared Errors (MSE < 0.2)')
    plt.xlabel("Mean Squared Error")
    plt.ylabel("Number of Proteins")
    plt.tight_layout()
    show_readable_figures()


In [ ]:
# Cell: MAE < 0.3
threshold = 0.3
subset = protein_means.loc[protein_means["mean_squared_error"] < threshold, "mean_squared_error"]

print(f"Proteins with mean_squared_error < {threshold}: {subset.shape[0]}")

if subset.empty:
    print("No proteins below this threshold to plot.")
else:
    plt.figure(figsize=readable_figsize((10, 5)))
    sns.histplot(subset, bins=30, kde=True, color="steelblue")
    _remember_figure_name(plt.gcf(), 'Distribution of Mean Squared Errors (MSE < 0.3)')
    plt.xlabel("Mean Squared Error")
    plt.ylabel("Number of Proteins")
    plt.tight_layout()
    show_readable_figures()


#### medians

In [ ]:
# Compute median squared error for each protein across samples
protein_medians = combined_df.groupby("protein")["squared_error"].median().reset_index()
protein_medians.rename(columns={"squared_error": "median_squared_error"}, inplace=True)

# Plot the distribution
plt.figure(figsize=readable_figsize((10, 5)))
sns.histplot(protein_medians["median_squared_error"], bins=30, kde=True, color="steelblue")
_remember_figure_name(plt.gcf(), 'Distribution of Median Squared Errors Across Proteins')
plt.xlabel("Median Squared Error")
plt.ylabel("Number of Proteins")
plt.tight_layout()
show_readable_figures()

In [ ]:
# MAE < 0.05
threshold = 0.05

# mask + subset for plotting
mask = protein_medians["median_squared_error"] < threshold
subset = protein_medians.loc[mask, "median_squared_error"]

print(f"Proteins with median_squared_error < {threshold}: {subset.shape[0]}")

if subset.empty:
    print("No proteins below this threshold to plot.")
else:
    # --- histogram (seaborn) ---
    plt.figure(figsize=readable_figsize((10, 5)))
    sns.histplot(subset, bins=30, kde=True, color="steelblue")
    _remember_figure_name(plt.gcf(), f'Distribution of Median Squared Errors < {threshold}')
    plt.xlabel("Median Squared Error")
    plt.ylabel("Number of Proteins")
    plt.tight_layout()
    show_readable_figures()

    # --- table of proteins with MSE < threshold ---
    # If there's a 'protein' column, use it; otherwise, lift the index into a column.
    if "protein" in protein_medians.columns:
        table = protein_medians.loc[mask, ["protein", "median_squared_error"]].copy()
    else:
        table = (
            protein_medians.loc[mask, ["median_squared_error"]]
            .reset_index()
            .rename(columns={"index": "protein"})
        )

    # sort for readability and format
    table = table.sort_values("median_squared_error").reset_index(drop=True)
    table["median_squared_error"] = table["median_squared_error"].astype(float)

    # --- SAVE TABLE ---
    from pathlib import Path
    from datetime import datetime

    out_dir = Path("exports")
    out_dir.mkdir(parents=True, exist_ok=True)

    stamp = datetime.now().strftime("%Y%m%d-%H%M%S")
    base = f"proteins_median_squared_error_lt_{threshold}".replace(".", "p")

    csv_path = out_dir / f"{base}_{stamp}.csv"
    

    table.to_csv(csv_path, index=False)
    print(f"Saved CSV -> {csv_path.resolve()}")

    # --- display nicely in notebooks ---
    try:
        from IPython.display import display
        print("\nTable: Proteins with Median Squared Error < {:.3f} (sorted ascending)".format(threshold))
        display(table.style.format({"median_squared_error": "{:.6f}"}))
    except Exception:
        print("\nTable: Proteins with Median Squared Error < {:.3f} (sorted ascending)".format(threshold))
        print(table.to_string(index=False))


In [ ]:
# MAE < 0.1
threshold = 0.1

# mask + subset for plotting
mask = protein_medians["median_squared_error"] < threshold
subset = protein_medians.loc[mask, "median_squared_error"]

print(f"Proteins with median_squared_error < {threshold}: {subset.shape[0]}")

if subset.empty:
    print("No proteins below this threshold to plot.")
else:
    # --- histogram (seaborn) ---
    plt.figure(figsize=readable_figsize((10, 5)))
    sns.histplot(subset, bins=30, kde=True, color="steelblue")
    _remember_figure_name(plt.gcf(), f'Distribution of Median Squared Errors < {threshold}')
    plt.xlabel("Median Squared Error")
    plt.ylabel("Number of Proteins")
    plt.tight_layout()
    show_readable_figures()

    # --- table of proteins with MSE < threshold ---
    # If there's a 'protein' column, use it; otherwise, lift the index into a column.
    if "protein" in protein_medians.columns:
        table = protein_medians.loc[mask, ["protein", "median_squared_error"]].copy()
    else:
        table = (
            protein_medians.loc[mask, ["median_squared_error"]]
            .reset_index()
            .rename(columns={"index": "protein"})
        )

    # sort for readability and format
    table = table.sort_values("median_squared_error").reset_index(drop=True)
    table["median_squared_error"] = table["median_squared_error"].astype(float)

    # --- SAVE TABLE ---
    from pathlib import Path
    from datetime import datetime

    out_dir = Path("exports")
    out_dir.mkdir(parents=True, exist_ok=True)

    stamp = datetime.now().strftime("%Y%m%d-%H%M%S")
    base = f"proteins_median_squared_error_lt_{threshold}".replace(".", "p")

    csv_path = out_dir / f"{base}_{stamp}.csv"
    

    table.to_csv(csv_path, index=False)
    print(f"Saved CSV -> {csv_path.resolve()}")

    # --- display nicely in notebooks ---
    try:
        from IPython.display import display
        print("\nTable: Proteins with Median Squared Error < {:.3f} (sorted ascending)".format(threshold))
        display(table.style.format({"median_squared_error": "{:.6f}"}))
    except Exception:
        print("\nTable: Proteins with Median Squared Error < {:.3f} (sorted ascending)".format(threshold))
        print(table.to_string(index=False))


In [ ]:
# Cell: MAE < 0.2
threshold = 0.2
subset = protein_medians.loc[protein_medians["median_squared_error"] < threshold, "median_squared_error"]

print(f"Proteins with median_squared_error < {threshold}: {subset.shape[0]}")

if subset.empty:
    print("No proteins below this threshold to plot.")
else:
    plt.figure(figsize=readable_figsize((10, 5)))
    sns.histplot(subset, bins=30, kde=True, color="steelblue")
    _remember_figure_name(plt.gcf(), 'Distribution of Median Squared Errors (MSE < 0.2)')
    plt.xlabel("Median Squared Error")
    plt.ylabel("Number of Proteins")
    plt.tight_layout()
    show_readable_figures()

In [ ]:
# Cell: MAE < 0.3
threshold = 0.3
subset = protein_medians.loc[protein_medians["median_squared_error"] < threshold, "median_squared_error"]

print(f"Proteins with median_squared_error < {threshold}: {subset.shape[0]}")

if subset.empty:
    print("No proteins below this threshold to plot.")
else:
    plt.figure(figsize=readable_figsize((10, 5)))
    sns.histplot(subset, bins=30, kde=True, color="steelblue")
    _remember_figure_name(plt.gcf(), 'Distribution of Median Squared Errors (MSE < 0.3)')
    plt.xlabel("Median Squared Error")
    plt.ylabel("Number of Proteins")
    plt.tight_layout()
    show_readable_figures()


### Other per-protein plots

In [ ]:
# MSE vs expression with robust limits and readable 1:1 line

# Recompute per-protein summary (safe if it already exists)
prot_perf = (
    combined_df.groupby('protein')
    .agg(expression=('true', 'mean'),
         mse=('squared_error', 'mean'),
         mae=('abs_error', 'mean'))
    .reset_index()
)

x = prot_perf['expression'].to_numpy()
y = prot_perf['mse'].to_numpy()

# Robust axis limits (clip to 0.5–99.5 percentiles and pad 5%)
x_lo, x_hi = np.nanpercentile(x, [0.5, 99.5])
y_lo, y_hi = np.nanpercentile(y, [0.5, 99.5])
x_pad = 0.05 * (x_hi - x_lo) if np.isfinite(x_hi - x_lo) else 1.0
y_pad = 0.05 * (y_hi - y_lo) if np.isfinite(y_hi - y_lo) else 1.0
xlim = (x_lo - x_pad, x_hi + x_pad)
ylim = (y_lo - y_pad, y_hi + y_pad)

plt.figure(figsize=readable_figsize((8, 6)))
sns.scatterplot(
    data=prot_perf, x='expression', y='mse',
    s=18, alpha=0.5, linewidth=0, edgecolor=None
)


plt.xlim(*xlim)
plt.ylim(*ylim)
plt.xlabel('Mean expression (true)')
plt.ylabel('Mean squared error (MSE)')
_remember_figure_name(plt.gcf(), 'Protein-wise: MSE vs Expression (robust limits, α=0.5)')
plt.grid(True, linestyle=':', linewidth=0.7, alpha=0.6)
plt.tight_layout()
show_readable_figures()


In [ ]:
# Mean and variance per protein
protein_stats = combined_df.groupby('protein')['abs_error'].agg(['mean', 'var']).reset_index()

# Plot
plt.figure(figsize=readable_figsize((10, 6)))
sns.scatterplot(data=protein_stats, x='mean', y='var')
_remember_figure_name(plt.gcf(), 'Mean vs Variance of Absolute Error per Protein')
plt.xlabel("Mean Absolute Error")
plt.ylabel("Variance of Absolute Error")
plt.grid(True)
show_readable_figures()


#### mean error vs variance of true values

In [ ]:
#Aggregate per protein 

prot_stats = (
    combined_df
    .groupby('protein')
    .agg(
        mean_error = ('abs_error', 'mean'),
        var_true   = ('true', 'var'),
    )
    .reset_index()
)

plt.figure(figsize=readable_figsize((8, 6)))
sns.scatterplot(
    data=prot_stats,
    x='var_true',
    y='mean_error',
    s=25,
    edgecolor='k'
)
plt.axhline(0, color='gray', linestyle='--', linewidth=1)
_remember_figure_name(plt.gcf(), 'Protein‑wise: Mean Prediction Error vs Variance of True Values')
plt.xlabel('Variance of True Values')
plt.ylabel('Mean Prediction Error')
plt.tight_layout()
show_readable_figures()

### Cosine similaity 

In [ ]:
# Top 50, vertical bars, steelblue gradient
# prep
df = per_protein.copy()
if 'protein' in df.columns:
    df = df.set_index('protein')

df_top = (
    df[['cosine_similarity']]
    .dropna()
    .sort_values('cosine_similarity', ascending=False)
    .head(50)
)

vals   = df_top['cosine_similarity'].to_numpy()
labels = df_top.index.astype(str)

# custom steelblue gradient (deep navy → steelblue → pale)
steelblue_cmap = LinearSegmentedColormap.from_list(
    "steelblue_ramp",
    ["#0f2d45", "#1f4e79", "#4682B4", "#9ab6d1", "#e2eef7"]
)

# color by rank so the gradient flows left→right
colors = steelblue_cmap(np.linspace(0.07, 0.97, len(vals)))

fig, ax = plt.subplots(figsize=readable_figsize((18, 6)), dpi=140)

x = np.arange(len(vals))
ax.bar(x, vals, color=colors, width=0.85)

# labels & limits
_remember_axes_name(ax, 'Top 50 proteins by cosine similarity')
ax.set_ylabel('Cosine similarity')
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=90, fontsize=17.4)
ax.set_ylim(0, max(1.0, vals.max() + 0.05))

# subtle grid & cleanup
ax.grid(False, which='both')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
show_readable_figures()


How many prteins after applying cosine similarity threshold of 0.5, 0.6, 0.7, 0.8, 0.9

In [ ]:
# Summary table for cosine similarity thresholds + per-threshold tables (no saving)

df = per_protein.copy()
if "protein" in df.columns:
    df = df.set_index("protein")

if "cosine_similarity" not in df.columns:
    raise KeyError("Expected a 'cosine_similarity' column in per_protein.")

# total rows considered (robust to NaNs)
cs = df["cosine_similarity"]
total = int(cs.notna().sum())

thresholds = [0.5, 0.6, 0.7, 0.8]
rows = []

def make_table_above_threshold(frame: pd.DataFrame, thresh: float) -> pd.DataFrame:
    cs_local = frame["cosine_similarity"]
    mask = cs_local.notna() & (cs_local > thresh)
    tbl = (
        frame.loc[mask, ["cosine_similarity"]]
             .sort_values("cosine_similarity", ascending=False)
             .assign(protein=lambda d: d.index)
             .loc[:, ["protein", "cosine_similarity"]]
             .reset_index(drop=True)
    )
    return tbl

# build tables
table05 = make_table_above_threshold(df, 0.5)
table06 = make_table_above_threshold(df, 0.6)
table07 = make_table_above_threshold(df, 0.7)
table08 = make_table_above_threshold(df, 0.8)

# summary counts/percents
for t in thresholds:
    n = int((cs.notna() & (cs > t)).sum())
    pct = (n / total * 100) if total else 0.0
    rows.append({"threshold": f">{t:.1f}", "n_proteins": n, "% of total": f"{pct:.1f}%"})

summary = pd.DataFrame(rows).set_index("threshold")

print(f"Total proteins considered: {total}\n")
print(summary.to_string())

# quick confirmation of table sizes
print("\nTables created (rows):",
      f"table05={len(table05)}",
      f"table06={len(table06)}",
      f"table07={len(table07)}",
      f"table08={len(table08)}")


In [ ]:
#positive 
pos = df[df['cosine_similarity'] > 0]
len(pos)

In [ ]:


# total rows considered (robust to NaNs)
total = int(pos['cosine_similarity'].notna().sum())

thresholds = [0.5, 0.6, 0.7, 0.8]
rows = []
for t in thresholds:
    n = int((pos['cosine_similarity'] > t).sum())
    pct = (n / total * 100) if total else 0.0
    rows.append({"threshold": f">{t:.1f}", "n_proteins": n, "% of total": f"{pct:.1f}%"})

summary = pd.DataFrame(rows).set_index("threshold")

print(f"Total proteins considered: {total}\n")
print(summary.to_string())


In [ ]:
df[['cosine_similarity']].describe()

# Samples analysis


### Top 100 samples by mean squared error, cosine similarity, pearson correlation

In [ ]:
# Per-sample metrics from combined_df → samples_df (369 rows expected)
eps = 1e-12

combined_df = combined_df.copy()

# row-wise errors
combined_df['abs_error']     = (combined_df['true'] - combined_df['predicted']).abs()
combined_df['squared_error'] = (combined_df['true'] - combined_df['predicted'])**2
combined_df['rel_error']     = combined_df['abs_error'] / (combined_df['true'].abs() + eps)

def _cos_sim(a, b):
    na = np.linalg.norm(a); nb = np.linalg.norm(b)
    denom = na * nb
    return float(np.dot(a, b) / denom) if denom > 0 else np.nan

# the column holding sample IDs (handles the space in "LOO sample")
sample_col = 'LOO sample' if 'LOO sample' in combined_df.columns else 'sample'

samples_df = (
    combined_df.groupby(sample_col, sort=False)
      .apply(lambda g: pd.Series({
          'n_proteins': len(g),
          'mae': g['abs_error'].mean(),
          'mse': g['squared_error'].mean(),
          'rmse': np.sqrt(g['squared_error'].mean()),
          'cosine_similarity': _cos_sim(g['true'].to_numpy(), g['predicted'].to_numpy()),
          'pearson_corr': g['true'].corr(g['predicted'])
      }))
      .reset_index()
      .rename(columns={sample_col: 'sample'})
)

samples_df.head()


### Top 100 proteins by mean squared error, cosine similarity, pearson correlation

In [ ]:

top100_mse     = samples_df.sort_values('mse', ascending=True).head(100)               # lower is better
top100_mae     = samples_df.sort_values('mae', ascending=True).head(100)               # optional: also show by MAE
top100_cosine  = samples_df.sort_values('cosine_similarity', ascending=False).head(100)
top100_pearson = samples_df.sort_values('pearson_corr',       ascending=False).head(100)

print("Top 100 samples by lowest MSE")
display(top100_mse)

print("\nTop 100 samples by lowest MAE")
display(top100_mae)

print("\nTop 100 samples by highest cosine similarity")
display(top100_cosine)

print("\nTop 100 samples by highest Pearson correlation")
display(top100_pearson)


In [ ]:
sample_error_summary = combined_df.groupby('LOO sample').agg(
    mean_abs_error=('abs_error', 'mean'),
    std_abs_error=('abs_error', 'std'),
    mean_squared_error=('squared_error', 'mean'),
    std_squared_error=('squared_error', 'std'),
    count=('LOO sample', 'count')
).sort_values(by='mean_abs_error', ascending=False)

sample_error_summary.head()

### MSE thresholding and distributions

In [ ]:
# Compute median absolute error across proteins for each sample
sample_means = combined_df.groupby("LOO sample")["squared_error"].mean().reset_index()
sample_means.rename(columns={"squared_error": "mean_squared_error"}, inplace=True)

# Plot
plt.figure(figsize=readable_figsize((10, 5)))
sns.histplot(sample_means["mean_squared_error"], bins=30, kde=True, color="mediumseagreen")
_remember_figure_name(plt.gcf(), 'Distribution of Mean Squared Errors Across Samples')
plt.xlabel("Mean Squared Error")
plt.ylabel("Number of Samples")
plt.tight_layout()
show_readable_figures()


In [ ]:
# Mean Squared Error per sample + threshold plot (< 0.1)

# column holding sample IDs (handles the space in "LOO sample")
sample_col = "LOO sample" if "LOO sample" in combined_df.columns else "sample"

# mean squared error across proteins for each sample
sample_means = (
    combined_df.groupby(sample_col)["squared_error"]
               .mean()
               .reset_index()
               .rename(columns={sample_col: "sample", "squared_error": "mean_squared_error"})
)

threshold = 0.1

# mask + subset for plotting
mask = sample_means["mean_squared_error"] < threshold
subset = sample_means.loc[mask, "mean_squared_error"]

print(f"Samples with mean_squared_error < {threshold}: {subset.shape[0]}")

if subset.empty:
    print("No samples below this threshold to plot.")
else:
    # --- histogram (seaborn) ---
    plt.figure(figsize=readable_figsize((10, 5)))
    sns.histplot(subset, bins=30, kde=True, color="mediumseagreen")
    _remember_figure_name(plt.gcf(), f'Distribution of Sample Mean Squared Errors (MSE < {threshold})')
    plt.xlabel("Mean Squared Error")
    plt.ylabel("Number of Samples")
    plt.tight_layout()
    show_readable_figures()

    # --- table of samples with MSE < threshold ---
    table = (
        sample_means.loc[mask, ["sample", "mean_squared_error"]]
        .sort_values("mean_squared_error")
        .reset_index(drop=True)
    )

    try:
        from IPython.display import display
        print("\nTable: Samples with MSE < {:.3f} (sorted ascending)".format(threshold))
        display(table.style.format({"mean_squared_error": "{:.6f}"}))
    except Exception:
        print("\nTable: Samples with MSE < {:.3f} (sorted ascending)".format(threshold))
        print(table.to_string(index=False))


In [ ]:
threshold = 0.2

mask = sample_means["mean_squared_error"] < threshold
subset = sample_means.loc[mask, "mean_squared_error"]

print(f"Samples with mean_squared_error < {threshold}: {subset.shape[0]}")

if subset.empty:
    print("No samples below this threshold to plot.")
else:
    plt.figure(figsize=readable_figsize((10, 5)))
    sns.histplot(subset, bins=30, kde=True, color="mediumseagreen")
    _remember_figure_name(plt.gcf(), f'Distribution of Sample Mean Squared Errors (MSE < {threshold})')
    plt.xlabel("Mean Squared Error")
    plt.ylabel("Number of Samples")
    plt.tight_layout()
    show_readable_figures()

    table = (
        sample_means.loc[mask, ["sample", "mean_squared_error"]]
        .sort_values("mean_squared_error")
        .reset_index(drop=True)
    )
    try:
        from IPython.display import display
        print("\nTable: Samples with MSE < {:.3f} (sorted ascending)".format(threshold))
        #display(table.style.format({"mean_squared_error": "{:.6f}"}))
    except Exception:
        print("\nTable: Samples with MSE < {:.3f} (sorted ascending)".format(threshold))
        print(table.to_string(index=False))


In [ ]:
threshold = 0.3

mask = sample_means["mean_squared_error"] < threshold
subset = sample_means.loc[mask, "mean_squared_error"]

print(f"Samples with mean_squared_error < {threshold}: {subset.shape[0]}")

if subset.empty:
    print("No samples below this threshold to plot.")
else:
    plt.figure(figsize=readable_figsize((10, 5)))
    sns.histplot(subset, bins=30, kde=True, color="mediumseagreen")
    _remember_figure_name(plt.gcf(), f'Distribution of Sample Mean Squared Errors (MSE < {threshold})')
    plt.xlabel("Mean Squared Error")
    plt.ylabel("Number of Samples")
    plt.tight_layout()
    show_readable_figures()

    table = (
        sample_means.loc[mask, ["sample", "mean_squared_error"]]
        .sort_values("mean_squared_error")
        .reset_index(drop=True)
    )
    try:
        from IPython.display import display
        print("\nTable: Samples with MSE < {:.3f} (sorted ascending)".format(threshold))
        #display(table.style.format({"mean_squared_error": "{:.6f}"}))
    except Exception:
        print("\nTable: Samples with MSE < {:.3f} (sorted ascending)".format(threshold))
        print(table.to_string(index=False))


#### Medians

In [ ]:
# Compute median absolute error across proteins for each sample
sample_medians = combined_df.groupby("LOO sample")["squared_error"].median().reset_index()
sample_medians.rename(columns={"squared_error": "median_squared_error"}, inplace=True)

# Plot
plt.figure(figsize=readable_figsize((10, 5)))
sns.histplot(sample_medians["median_squared_error"], bins=30, kde=True, color="mediumseagreen")
_remember_figure_name(plt.gcf(), 'Distribution of Median Squared Errors Across Samples')
plt.ylabel("Number of Samples")
plt.tight_layout()
show_readable_figures()


In [ ]:
# Mean Squared Error per sample + threshold plot (< 0.05)

# column holding sample IDs (handles the space in "LOO sample")
sample_col = "LOO sample" if "LOO sample" in combined_df.columns else "sample"

threshold = 0.1

# mask + subset for plotting
mask = sample_medians["median_squared_error"] < threshold
subset = sample_medians.loc[mask, "median_squared_error"]

print(f"Samples with median_squared_error < {threshold}: {subset.shape[0]}")

if subset.empty:
    print("No samples below this threshold to plot.")
else:
    # --- histogram (seaborn) ---
    plt.figure(figsize=readable_figsize((10, 5)))
    sns.histplot(subset, bins=30, kde=True, color="mediumseagreen")
    _remember_figure_name(plt.gcf(), f'Distribution of Sample Mean Squared Errors (MSE < {threshold})')
    plt.xlabel("Mean Squared Error")
    plt.ylabel("Number of Samples")
    plt.tight_layout()
    show_readable_figures()

    # --- table of samples with MSE < threshold ---
    table = (
        sample_medians.loc[mask, [sample_col, "median_squared_error"]]
        .sort_values("median_squared_error")
        .reset_index(drop=True)
    )

    try:
        from IPython.display import display
        print("\nTable: Samples with Median Squared Error < {:.3f} (sorted ascending)".format(threshold))
        display(table.style.format({"median_squared_error": "{:.6f}"}))
    except Exception:
        print("\nTable: Samples with Median Squared Error < {:.3f} (sorted ascending)".format(threshold))
        print(table.to_string(index=False))


In [ ]:
# Mean Squared Error per sample + threshold plot (< 0.05)

# column holding sample IDs (handles the space in "LOO sample")
sample_col = "LOO sample" if "LOO sample" in combined_df.columns else "sample"

threshold = 0.05

# mask + subset for plotting
mask = sample_medians["median_squared_error"] < threshold
subset = sample_medians.loc[mask, "median_squared_error"]

print(f"Samples with median_squared_error < {threshold}: {subset.shape[0]}")

if subset.empty:
    print("No samples below this threshold to plot.")
else:
    # --- histogram (seaborn) ---
    plt.figure(figsize=readable_figsize((10, 5)))
    sns.histplot(subset, bins=30, kde=True, color="mediumseagreen")
    _remember_figure_name(plt.gcf(), f'Distribution of Sample Mean Squared Errors (MSE < {threshold})')
    plt.xlabel("Mean Squared Error")
    plt.ylabel("Number of Samples")
    plt.tight_layout()
    show_readable_figures()

    # --- table of samples with MSE < threshold ---
    table = (
        sample_medians.loc[mask, [sample_col, "median_squared_error"]]
        .sort_values("median_squared_error")
        .reset_index(drop=True)
    )

    try:
        from IPython.display import display
        print("\nTable: Samples with Median Squared Error < {:.3f} (sorted ascending)".format(threshold))
        display(table.style.format({"median_squared_error": "{:.6f}"}))
    except Exception:
        print("\nTable: Samples with Median Squared Error < {:.3f} (sorted ascending)".format(threshold))
        print(table.to_string(index=False))


In [ ]:
# Mean Squared Error per sample + threshold plot (< 0.1)

# column holding sample IDs (handles the space in "LOO sample")
sample_col = "LOO sample" if "LOO sample" in combined_df.columns else "sample"

threshold = 0.1

# mask + subset for plotting
mask = sample_medians["median_squared_error"] < threshold
subset = sample_medians.loc[mask, "median_squared_error"]

print(f"Samples with median_squared_error < {threshold}: {subset.shape[0]}")

if subset.empty:
    print("No samples below this threshold to plot.")
else:
    # --- histogram (seaborn) ---
    plt.figure(figsize=readable_figsize((10, 5)))
    sns.histplot(subset, bins=30, kde=True, color="mediumseagreen")
    _remember_figure_name(plt.gcf(), f'Distribution of Sample Mean Squared Errors (MSE < {threshold})')
    plt.xlabel("Mean Squared Error")
    plt.ylabel("Number of Samples")
    plt.tight_layout()
    show_readable_figures()

    # --- table of samples with MSE < threshold ---
    table = (
        sample_medians.loc[mask, [sample_col, "median_squared_error"]]
        .sort_values("median_squared_error")
        .reset_index(drop=True)
    )

    try:
        from IPython.display import display
        print("\nTable: Samples with Median Squared Error < {:.3f} (sorted ascending)".format(threshold))
        display(table.style.format({"median_squared_error": "{:.6f}"}))
    except Exception:
        print("\nTable: Samples with Median Squared Error < {:.3f} (sorted ascending)".format(threshold))
        print(table.to_string(index=False))


In [ ]:
threshold = 0.2

mask = sample_medians["median_squared_error"] < threshold
subset = sample_medians.loc[mask, "median_squared_error"]

print(f"Samples with median_squared_error < {threshold}: {subset.shape[0]}")

if subset.empty:
    print("No samples below this threshold to plot.")
else:
    plt.figure(figsize=readable_figsize((10, 5)))
    sns.histplot(subset, bins=30, kde=True, color="mediumseagreen")
    _remember_figure_name(plt.gcf(), f'Distribution of Sample Mean Squared Errors (MSE < {threshold})')
    plt.xlabel("Mean Squared Error")
    plt.ylabel("Number of Samples")
    plt.tight_layout()
    show_readable_figures()

    table = (
        sample_medians.loc[mask, [sample_col, "median_squared_error"]]
        .sort_values("median_squared_error")
        .reset_index(drop=True)
    )
    try:
        from IPython.display import display
        print("\nTable: Samples with Median Squared Error < {:.3f} (sorted ascending)".format(threshold))
        #display(table.style.format({"median_squared_error": "{:.6f}"}))
    except Exception:
        print("\nTable: Samples with Median Squared Error < {:.3f} (sorted ascending)".format(threshold))
        print(table.to_string(index=False))


In [ ]:
threshold = 0.3

mask = sample_medians["median_squared_error"] < threshold
subset = sample_medians.loc[mask, "median_squared_error"]

print(f"Samples with median_squared_error < {threshold}: {subset.shape[0]}")

if subset.empty:
    print("No samples below this threshold to plot.")
else:
    plt.figure(figsize=readable_figsize((10, 5)))
    sns.histplot(subset, bins=30, kde=True, color="mediumseagreen")
    _remember_figure_name(plt.gcf(), f'Distribution of Sample Mean Squared Errors (MSE < {threshold})')
    plt.xlabel("Mean Squared Error")
    plt.ylabel("Number of Samples")
    plt.tight_layout()
    show_readable_figures()

    table = (
        sample_medians.loc[mask, [sample_col, "median_squared_error"]]
        .sort_values("median_squared_error")
        .reset_index(drop=True)
    )
    try:
        from IPython.display import display
        print("\nTable: Samples with Median Squared Error < {:.3f} (sorted ascending)".format(threshold))
        #display(table.style.format({"median_squared_error": "{:.6f}"}))
    except Exception:
        print("\nTable: Samples with Median Squared Error < {:.3f} (sorted ascending)".format(threshold))
        print(table.to_string(index=False))


### Cosine similarity

In [ ]:
# Top 50 samples by cosine similarity — vertical bars, Medium SeaGreen gradient
# Build samples_df if it doesn't already exist
try:
    samples_df
except NameError:
    sample_col = "LOO sample" if "LOO sample" in combined_df.columns else "sample"
    if "squared_error" not in combined_df.columns:
        combined_df = combined_df.copy()
        combined_df["squared_error"] = (combined_df["true"] - combined_df["predicted"])**2

    samples_df = (
        clean.groupby(sample_col, sort=False)
        .apply(lambda g: pd.Series({
            "n_proteins": len(g),
            "mse": g["squared_error"].mean(),
            "cosine_similarity": _cos_sim(g["true"].to_numpy(), g["predicted"].to_numpy()),
            "pearson_corr": g["true"].corr(g["predicted"]),
        }))
        .reset_index()
        .rename(columns={sample_col: "sample"})
    )

# Sort & take top K
k = 50
df_top = (
    samples_df[["sample", "cosine_similarity"]]
    .dropna()
    .sort_values("cosine_similarity", ascending=False)
    .head(k)
)

vals   = df_top["cosine_similarity"].to_numpy()
labels = df_top["sample"].astype(str).to_numpy()

# Medium SeaGreen tonality (deep green → MediumSeaGreen → pale mint)
seagreen_cmap = LinearSegmentedColormap.from_list(
    "seagreen_ramp",
    ["#0f4d3a", "#2f8f68", "#3CB371", "#8ad9b2", "#d7f5e8"]  # tweak if we want it cooler/warmer
)
colors = seagreen_cmap(np.linspace(0.07, 0.97, len(vals)))  # smooth rank-based gradient

fig, ax = plt.subplots(figsize=readable_figsize((18, 6)), dpi=140)
x = np.arange(len(vals))
ax.bar(x, vals, color=colors, width=0.85)

# Labels & limits
_remember_axes_name(ax, 'Top 50 samples by cosine similarity')
ax.set_ylabel("Cosine similarity")
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=90, fontsize=11.6)
ax.set_ylim(0, max(1.0, vals.max() + 0.05))

# Tidy up
ax.grid(False, which='both')
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
show_readable_figures()


In [ ]:
s = samples_df['sample'].astype(str).str.strip()
parts = s.str.split('_', n=1, expand=True)     # use n=1 as a keyword
samples_df['tissue'] = parts[1].where(parts[1].notna() & (parts[1] != ''), parts[0])

# keep one sample per tissue: the one with the highest cosine
df_best = (samples_df.dropna(subset=['tissue','cosine_similarity'])
                     .sort_values('cosine_similarity', ascending=False)
                     .drop_duplicates(subset=['tissue'], keep='first'))

# data for plotting (ALL tissues)
vals   = df_best['cosine_similarity'].to_numpy()
labels = df_best['tissue'].astype(str).to_numpy()

# Medium SeaGreen gradient (deep → light)
seagreen_cmap = LinearSegmentedColormap.from_list(
    "seagreen_ramp", ["#0f4d3a", "#2f8f68", "#3CB371", "#8ad9b2", "#d7f5e8"]
)
colors = seagreen_cmap(np.linspace(0.07, 0.97, len(vals)))

# dynamic width to fit all tissues
fig_w = max(12, 0.35 * len(vals))
fig, ax = plt.subplots(figsize=readable_figsize((fig_w, 6)), dpi=130)

x = np.arange(len(vals))
ax.bar(x, vals, color=colors, width=0.85, edgecolor='none', linewidth=0)

_remember_axes_name(ax, 'Best sample per tissue by cosine similarity (all tissues)')
ax.set_ylabel("Cosine similarity")
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=60, ha='right', fontsize=13.0)
ax.set_ylim(0, max(1.0, float(vals.max()) + 0.05))

# clean look (no grid/edges)
ax.grid(False, which='both')
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
show_readable_figures()

In [ ]:
# total rows considered (robust to NaNs)
total = int(samples_df['cosine_similarity'].notna().sum())

thresholds = [0.5, 0.6, 0.7, 0.8]
rows = []
for t in thresholds:
    n = int((samples_df['cosine_similarity'] > t).sum())
    pct = (n / total * 100) if total else 0.0
    rows.append({"threshold": f">{t:.1f}", "n_samples": n, "% of total": f"{pct:.1f}%"})

summary = pd.DataFrame(rows).set_index("threshold")

print(f"Total Samples considered: {total}\n")
print(summary.to_string())


In [ ]:
# --- aggregate per tissue (MEAN)
tissue_table = (
    samples_df.dropna(subset=['tissue'])
              .groupby('tissue', as_index=False)
              .agg(n_samples=('sample', 'nunique'),
                   cosine_similarity=('cosine_similarity', 'mean'),
                   mse=('mse', 'mean'))
              .sort_values('cosine_similarity', ascending=False)
)

# --- display
try:
    display(
        tissue_table.style
            .format({'cosine_similarity': '{:.4f}', 'mse': '{:.6f}', 'n_samples': '{:d}'})
            .set_properties(subset=['tissue'], **{'font-weight': 'bold'})
    )
except Exception:
    print(tissue_table.to_string(index=False))

In [ ]:

# --- aggregate per tissue (median; switch to 'mean' if preferred)
tissue_stats = (
    samples_df.dropna(subset=['tissue', 'cosine_similarity'])
              .groupby('tissue')
              .agg(cosine_similarity=('cosine_similarity', 'median'),
                   n_samples=('sample', 'nunique'))
              .sort_values('cosine_similarity', ascending=False)
)

vals   = tissue_stats['cosine_similarity'].to_numpy()
labels = tissue_stats.index.astype(str).to_numpy()

# Light purple (lilac) gradient
lilac_cmap = LinearSegmentedColormap.from_list(
    "lilac_ramp",
    ["#5b4b8a", "#7e6db1", "#a896e0", "#cdbbf0", "#f0e9ff"] 
)
colors = lilac_cmap(np.linspace(0.07, 0.97, len(vals)))

# dynamic width for all tissues
fig_w = max(12, 0.35 * len(vals))
fig, ax = plt.subplots(figsize=readable_figsize((fig_w, 6)), dpi=130)

x = np.arange(len(vals))
ax.bar(x, vals, color=colors, width=0.85, edgecolor='none', linewidth=0)

_remember_axes_name(ax, 'Tissue-level cosine similarity (median across samples)')
ax.set_ylabel("Cosine similarity")
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=60, ha='right', fontsize=13.0)
ax.set_ylim(0, max(1.0, float(vals.max()) + 0.05))

ax.set_axisbelow(True)
ax.grid(False, which='both')

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
show_readable_figures()

### Other statistical plots on Samples

In [ ]:
sample_stats = combined_df.groupby('LOO sample')['abs_error'].agg(['mean', 'var']).reset_index()

# Plot
plt.figure(figsize=readable_figsize((10, 6)))
sns.scatterplot(data=sample_stats, x='mean', y='var')
_remember_figure_name(plt.gcf(), 'Mean vs Variance of Absolute Error per LOO Sample')
plt.xlabel("Mean Absolute Error")
plt.ylabel("Variance of Absolute Error")
plt.grid(True)
show_readable_figures()


In [ ]:
# Aggregate per LOO sample
sample_stats = (
    combined_df
    .groupby('LOO sample')
    .agg(
        mean_error = ('abs_error', 'mean'),
        var_true    = ('true', 'var'),
    )
    .reset_index()
)

plt.figure(figsize=readable_figsize((8, 6)))
sns.scatterplot(
    data=sample_stats,
    x='var_true',
    y='mean_error',
    s=25,
    edgecolor='k'
)
plt.axhline(0, color='gray', linestyle='--', linewidth=1)
_remember_figure_name(plt.gcf(), 'Sample‑wise: Mean Prediction Error vs Variance of True Values')
plt.xlabel('Variance of True Values')
plt.ylabel('Mean Prediction Error')
plt.tight_layout()
show_readable_figures()

## Boxplot Protein and Samples

In [ ]:
# Visually enhanced box/violin plot of MSE: proteins vs samples

df = combined_df.copy()
sample_col = "LOO sample" if "LOO sample" in df.columns else "sample"

# ensure squared_error exists
if "squared_error" not in df.columns:
    df["squared_error"] = (df["true"] - df["predicted"])**2

# per-group MSE
protein_mse = (
    df.groupby("protein", sort=False)["squared_error"]
      .mean()
      .reset_index()
      .rename(columns={"squared_error": "mse"})
)
sample_mse = (
    df.groupby(sample_col, sort=False)["squared_error"]
      .mean()
      .reset_index()
      .rename(columns={"squared_error": "mse", sample_col: "sample"})
)

# tidy for plotting
plot_df = pd.concat(
    [protein_mse.assign(group="Proteins")[["group", "mse"]],
     sample_mse.assign(group="Samples")[["group", "mse"]]],
    ignore_index=True
)

# aesthetics
palette = {"Proteins": "#4682B4",  # steelblue
           "Samples":  "#3CB371"}  # mediumseagreen
set_readable_theme(style="whitegrid", context="talk")

# y-limit trimmed to reduce extreme outlier pull
ymax = np.percentile(plot_df["mse"], 99.5)
ymax = float(np.clip(ymax * 1.05, 0.1, None))  # small buffer

fig, ax = plt.subplots(figsize=readable_figsize((8, 5)), dpi=110)  # was (18, 6), 140


# violin (soft fill, no inner lines)
sns.violinplot(
    data=plot_df, x="group", y="mse",
    palette=palette, inner=None, cut=0, width=0.9, alpha=0.6, ax=ax
)

# box overlay (thin, clean)
sns.boxplot(
    data=plot_df, x="group", y="mse",
    width=0.32, showcaps=True, showfliers=False, boxprops={"facecolor": "white", "alpha": 0.9},
    medianprops={"linewidth": 2, "color": "#222"},
    whiskerprops={"linewidth": 1.4},
    capprops={"linewidth": 1.4},
    ax=ax
)

# light jittered points (subsample to keep it crisp)
pts = (
    plot_df.groupby("group", group_keys=False)
           .apply(lambda g: g.sample(n=min(500, len(g)), random_state=0))
)
sns.stripplot(
    data=pts, x="group", y="mse",
    color="k", alpha=0.18, size=2, jitter=0.28, ax=ax, zorder=1
)

# titles, labels, and counts
counts = {"Proteins": len(protein_mse), "Samples": len(sample_mse)}
_remember_axes_name(ax, 'MSE distributions: proteins vs. samples')
ax.set_xlabel("")
ax.set_ylabel("Mean Squared Error (MSE)")
ax.set_xticklabels([
    f"Proteins (n={counts['Proteins']})",
    f"Samples (n={counts['Samples']})"
])

# median annotations
meds = plot_df.groupby("group")["mse"].median().to_dict()
for i, g in enumerate(["Proteins", "Samples"]):
    m = meds[g]
    ax.text(i, m, f"median {m:.3f}", ha="center", va="bottom",
            fontsize=13.0, weight="bold",
            bbox=dict(boxstyle="round,pad=0.25", facecolor="white", edgecolor="none", alpha=0.8))

# tidy up
ax.set_ylim(0, ymax)
ax.grid(axis="y", linestyle="--", alpha=0.25)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
show_readable_figures()


# MAE vs Prot- Encoding Gene correlations

In [ ]:
corrs

## Final compact XGBoost protein metrics: 1 × 3

Protein PCC, MSE, and cosine similarity from `per_protein_df`, the table used in the earlier detailed protein performance plot. The dashed red line marks the mean of each metric. Summary statistics and metric-specific best/worst proteins are printed below as tables.


In [ ]:
# Compact protein-wise distributions and separate data tables.
# These functions reuse the metric tables computed earlier in the notebook.
COMPACT_TABLE_DIR = FIGURES_PDF_DIR / 'tables'
COMPACT_TABLES = {}


def print_compact_protein_tables(protein_metrics, model_label, file_stem):
    """Print and save summary statistics and metric-specific best/worst proteins."""
    data = protein_metrics.copy()
    if 'protein' not in data.columns:
        if data.index.name == 'protein':
            data = data.reset_index()
        else:
            raise ValueError(f'{model_label}: expected a protein column.')
    metric_specs = [
        ('pearson', 'PCC', True),
        ('mse', 'MSE', False),
        ('cosine', 'Cosine similarity', True),
    ]
    required = ['protein'] + [metric for metric, _, _ in metric_specs]
    missing = [column for column in required if column not in data.columns]
    if missing:
        raise ValueError(f'{model_label}: missing columns {missing}')

    summary_rows = []
    rankings = {}
    for metric, display_name, higher_is_better in metric_specs:
        values = pd.to_numeric(data[metric], errors='coerce').replace([np.inf, -np.inf], np.nan)
        data[metric] = values
        valid = values.dropna()
        summary_rows.append({
            'metric': display_name,
            'n_valid': len(valid),
            'n_missing_or_nonfinite': len(values) - len(valid),
            'mean': valid.mean(),
            'std': valid.std(),
            'min': valid.min(),
            'q25': valid.quantile(0.25),
            'median': valid.median(),
            'q75': valid.quantile(0.75),
            'max': valid.max(),
        })
        ordered = data.loc[values.notna(), ['protein', metric]].sort_values(
            metric, ascending=not higher_is_better, kind='stable',
        )
        best = ordered.head(10).copy()
        worst = ordered.sort_values(metric, ascending=higher_is_better, kind='stable').head(10).copy()
        best.insert(0, 'set', 'Best')
        worst.insert(0, 'set', 'Worst')
        best.insert(1, 'rank', np.arange(1, len(best) + 1))
        worst.insert(1, 'rank', np.arange(1, len(worst) + 1))
        rankings[metric] = pd.concat([best, worst], ignore_index=True)

    summary = pd.DataFrame(summary_rows)
    COMPACT_TABLE_DIR.mkdir(parents=True, exist_ok=True)
    summary.to_csv(COMPACT_TABLE_DIR / f'{file_stem}_summary_statistics.csv', index=False)
    print('\n' + '=' * 80)
    print(model_label)
    print('Summary statistics:')
    print(summary.to_string(index=False, float_format=lambda value: f'{value:.6g}'))
    for metric, display_name, _ in metric_specs:
        table = rankings[metric]
        print(f'\nBest / worst proteins by {display_name}:')
        print(table.to_string(index=False, float_format=lambda value: f'{value:.6g}'))
        table.to_csv(COMPACT_TABLE_DIR / f'{file_stem}_best_worst_{metric}.csv', index=False)
    result = {'summary': summary, 'rankings': rankings}
    COMPACT_TABLES[file_stem] = result
    return result


def plot_compact_protein_metrics(protein_metrics, model_label, file_stem):
    """One title-free row: protein PCC, MSE and cosine-similarity distributions."""
    print_compact_protein_tables(protein_metrics, model_label, file_stem)
    colors = _get_group_colors('proteins')
    fig, axes = plt.subplots(1, 3, figsize=(15.5, 2.6))
    fig.patch.set_facecolor('white')
    fig._compact_protein_metrics = True
    fig._readable_pdf_pad = 0.04
    fig._readable_export_name = f'{file_stem}.pdf'
    for column, (metric, xlabel) in enumerate([
        ('pearson', 'Protein PCC'),
        ('mse', 'Protein MSE'),
        ('cosine', 'Protein cosine similarity'),
    ]):
        ax = axes[column]
        values = pd.to_numeric(protein_metrics[metric], errors='coerce')
        values = values[np.isfinite(values)]
        if len(values):
            ax.hist(values, bins=30, color=colors[metric], edgecolor='black', linewidth=0.6, alpha=0.85)
            ax.axvline(values.mean(), color='red', linestyle='--', linewidth=1.6)
        else:
            ax.text(0.5, 0.5, 'No finite values', transform=ax.transAxes,
                    ha='center', va='center', fontsize=16)
        ax.set_xlabel(xlabel, fontsize=18, labelpad=5)
        ax.set_ylabel('Proteins' if column == 0 else '', fontsize=17, labelpad=5)
        ax.tick_params(axis='both', labelsize=15.5, pad=3)
        ax.xaxis.set_major_locator(MaxNLocator(nbins=4))
        ax.yaxis.set_major_locator(MaxNLocator(nbins=3, integer=True))
        ax.grid(axis='y', linestyle='--', alpha=0.20)
        ax.set_axisbelow(True)
        ax.set_title('')
    prepare_readable_figure(fig)
    _cache_figure_pdf(fig, source_name=file_stem)
    show_readable_figures()
    # The in-memory PDF is retained for the last export cell.
    plt.close(fig)
    return fig


In [ ]:
compact_xgb_figure = plot_compact_protein_metrics(
    per_protein_df,
    model_label='XGBoost LOO protein-wise metrics',
    file_stem='xgboost_loo_protein_metrics_1x3',
)
print('Compact table CSV files:', COMPACT_TABLE_DIR)


## Save all figures to PDF

Run this last cell to save every captured figure into `figures_pdf` beside this notebook. The compact 1 × 3 figure is saved as `xgboost_loo_protein_metrics_1x3.pdf`.


In [ ]:
pdf_files = export_notebook_figures()
pdf_files
